# M2BERT Fine-tuning for Pitch Correction

Compare pretrained **M2BERT** (ModernBERT-base backbone, CP compound-token representation)
against the custom Transformer **v5** on the OLiMPiC pitch-correction task.

**Setup**: freeze all but last 2 transformer layers (10, 11) + new 52-class head.  
**Budget**: ≤ 3 epochs **or** ≤ 10 minutes.  
**Baseline**: v5 dev top-1 = 83/499 (16.63%), top-5 = 261/499 (52.30%).


In [1]:
import sys, json, re, random, time, pickle
from pathlib import Path
from fractions import Fraction as _Frac
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ModernBertConfig, ModernBertModel


In [2]:
ML_DIR       = Path('/Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml')
PROCESSED    = ML_DIR / 'data' / 'processed'
PRETRAINED   = ML_DIR / 'data' / 'pretrained' / 'm2bert'
M2_RUN_DIR   = ML_DIR / 'data' / 'runs' / 'm2bert_finetune'
DATASET_ROOT = ML_DIR / 'dataset' / 'olimpic-1.0-synthetic'
M2_RUN_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cuda') if torch.cuda.is_available() else
          torch.device('cpu'))
print(f'DEVICE: {DEVICE}')

# ── CP vocab (from M2BERT pretrained) ──────────────────────────
with open(PRETRAINED / 'CP.pkl', 'rb') as f:
    _cp = pickle.load(f)

# _cp[0] = {attr_name: {token_str: id}}
# _cp[1] = {attr_idx: {id: token_str}}
CP_TOK2ID = _cp[0]
print('CP attribute sizes:', {k: len(v) for k, v in CP_TOK2ID.items()})

BAR_PAD    = CP_TOK2ID['Bar']['Bar <PAD>']           # 2
BAR_MASK   = CP_TOK2ID['Bar']['Bar <MASK>']          # 3
POS_PAD    = CP_TOK2ID['Position']['Position <PAD>'] # 16
PITCH_PAD  = CP_TOK2ID['Pitch']['Pitch <PAD>']       # 86
PITCH_MASK = CP_TOK2ID['Pitch']['Pitch <MASK>']      # 87
DUR_PAD    = CP_TOK2ID['Duration']['Duration <PAD>'] # 64

CP_PAD = [BAR_PAD, POS_PAD, PITCH_PAD, DUR_PAD]
print(f'CP_PAD: {CP_PAD}   PITCH_MASK: {PITCH_MASK}')


DEVICE: mps
CP attribute sizes: {'Bar': 4, 'Position': 18, 'Pitch': 88, 'Duration': 66}
CP_PAD: [2, 16, 86, 64]   PITCH_MASK: 87


In [3]:
PITCH_RE = re.compile(r'^[A-G][0-9]$')

def _lmx_path(sid):
    parts = sid.split('/')
    return DATASET_ROOT / 'samples' / parts[1] / (parts[2] + '.lmx')

# 52-class pitch mapping — identical to main notebook's build_pitch_classes()
pitches = set()
for sid in (DATASET_ROOT / 'samples.train.txt').read_text().split():
    p = _lmx_path(sid)
    if p.exists():
        pitches.update(t for t in p.read_text().split() if PITCH_RE.match(t))
pitch_classes = {p: i for i, p in enumerate(sorted(pitches))}
id2pitch      = {i: p for p, i in pitch_classes.items()}
N_CLASSES     = len(pitch_classes)
print(f'Pitch classes: {N_CLASSES}  (classes: {sorted(pitch_classes)[:5]} ... {sorted(pitch_classes)[-3:]})')

with open(PROCESSED / 'correction_train_10k.jsonl') as f:
    train_raw = [json.loads(l) for l in f]
with open(PROCESSED / 'correction_dev_v1.jsonl') as f:
    dev_raw = [json.loads(l) for l in f]
print(f'Train: {len(train_raw)}  Dev: {len(dev_raw)}')


Pitch classes: 52  (classes: ['A0', 'A1', 'A2', 'A3', 'A4'] ... ['G5', 'G6', 'G7'])
Train: 9998  Dev: 499


## LMX → CP Conversion

Each example's full token sequence is parsed into per-note dicts by `parse_lmx_notes`.
For each note we compute four CP indices:

| Attribute | Formula |
|-----------|---------|
| Bar       | 0 (Bar New) if note starts a new measure vs previous note, else 1 (Bar Continue) |
| Position  | `min(15, int(onset_in_measure_whole_notes × 16))` |
| Pitch     | `MIDI - 22` (0–85); 87 = PITCH_MASK for error note; 86 = PITCH_PAD for rest |
| Duration  | `min(63, int(duration_whole_notes × 32))`; 64 = DUR_PAD if unknown |

The full note sequence (no windowing needed; max 216 notes across all examples) is used.


In [4]:
# Note-Level LMX Parser — copied verbatim from correction_experiments.ipynb cell 41.
import re as _re

_DURATION_BASE = {
    'breve': _Frac(2),      'whole': _Frac(1),      'half': _Frac(1, 2),
    'quarter': _Frac(1, 4), 'eighth': _Frac(1, 8),  '16th': _Frac(1, 16),
    '32nd': _Frac(1, 32),   '64th': _Frac(1, 64),   '128th': _Frac(1, 128),
    '256th': _Frac(1, 256), '512th': _Frac(1, 512), '1024th': _Frac(1, 1024),
}
_DECORATION = frozenset({
    'beam:begin', 'beam:end', 'beam:forward-hook', 'beam:backward-hook',
    'tied:start', 'tied:stop', 'tuplet:start', 'tuplet:stop',
    'slur:start', 'slur:stop', 'trill-mark', 'staccato', 'accent',
    'strong-accent', 'tenuto', 'fermata', 'arpeggiate',
    'tremolo:1', 'tremolo:2', 'tremolo:3', 'tremolo:4',
    'tremolo:single', 'tremolo:start', 'tremolo:stop',
})
_ACCIDENTAL = frozenset({
    'sharp', 'flat', 'natural', 'double-sharp',
    'flat-flat', 'natural-sharp', 'natural-flat',
})
_SKIP = frozenset({'time'})
_TIME_MOD_RE = _re.compile(r'^(\d+)in(\d+)$')

def _parse_dur(tokens, i):
    if i >= len(tokens) or tokens[i] not in _DURATION_BASE:
        return None, i
    base = _DURATION_BASE[tokens[i]]
    i += 1
    m = _TIME_MOD_RE.match(tokens[i]) if i < len(tokens) else None
    if m:
        base *= _Frac(int(m.group(2)), int(m.group(1)))
        i += 1
    n_dots = 0
    while i < len(tokens) and tokens[i] == 'dot':
        n_dots += 1
        i += 1
    if n_dots:
        base *= (2 - _Frac(1, 2 ** n_dots))
    return base, i

def parse_lmx_notes(tokens, sample_id=''):
    notes, warnings = [], []
    current_voice = current_staff = None
    current_beats = current_beat_type = None
    measure_num = 0
    current_onset = _Frac(0)
    chord_group_id = 0
    chord_leader_onset = _Frac(0)
    i, n = 0, len(tokens)
    while i < n:
        tok = tokens[i]
        if tok == 'measure':
            measure_num += 1
            current_onset = _Frac(0)
            i += 1; continue
        if tok in _SKIP or tok.startswith('key:'):
            i += 1; continue
        if tok.startswith('beats:'):
            try: current_beats = int(tok[6:])
            except ValueError: warnings.append(f'bad beats {tok!r}')
            i += 1; continue
        if tok.startswith('beat-type:'):
            try: current_beat_type = int(tok[10:])
            except ValueError: warnings.append(f'bad beat-type {tok!r}')
            i += 1; continue
        if tok.startswith('clef:'):
            i += 1
            if i < n and tokens[i].startswith('staff:'): i += 1
            continue
        if tok in _DECORATION:
            i += 1; continue
        span_start = i
        is_grace = is_chord = False
        if tok == 'print-object:no':
            i += 1
            if i >= n: break
            tok = tokens[i]
        if tok == 'grace':
            is_grace = True; i += 1
            if i >= n: break
            tok = tokens[i]
            if tok == 'grace:slash':
                i += 1
                if i >= n: break
                tok = tokens[i]
        if tok == 'chord':
            is_chord = True; i += 1
            if i >= n: break
            tok = tokens[i]
        pitch = None; is_rest = False; pitch_token_position = None
        if PITCH_RE.match(tok):
            pitch, pitch_token_position = tok, i; i += 1
        elif tok == 'rest':
            is_rest = True; i += 1
        elif tok == 'backup':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset = max(_Frac(0), current_onset - dur)
            else: warnings.append(f'backup no dur @ {i}')
            continue
        elif tok == 'forward':
            i += 1; dur, i = _parse_dur(tokens, i)
            if dur is not None: current_onset += dur
            else: warnings.append(f'forward no dur @ {i}')
            continue
        else:
            warnings.append(f'unexpected {tok!r} @ {i}'); i += 1; continue
        if i < n and tokens[i].startswith('voice:'):
            current_voice = tokens[i]; i += 1
        duration = None
        if i < n:
            if tokens[i] in _DURATION_BASE:
                duration, i = _parse_dur(tokens, i)
            elif tokens[i] == 'rest:measure':
                if current_beats is not None and current_beat_type is not None:
                    duration = _Frac(current_beats, current_beat_type)
                else:
                    duration = None
                    warnings.append(f'rest:measure unknown dur @ {sample_id}')
                i += 1
        if i < n and tokens[i] in _ACCIDENTAL: i += 1
        if i < n and tokens[i].startswith('stem:'): i += 1
        if i < n and tokens[i].startswith('staff:'):
            current_staff = tokens[i]; i += 1
        while i < n and tokens[i] in _DECORATION: i += 1
        onset = chord_leader_onset if is_chord else current_onset
        if is_chord: grp = chord_group_id
        else:
            chord_group_id += 1; grp = chord_group_id
            chord_leader_onset = current_onset
        notes.append({'measure': measure_num, 'voice': current_voice, 'staff': current_staff,
                      'onset': onset, 'duration': duration, 'pitch': pitch,
                      'is_rest': is_rest, 'is_grace': is_grace,
                      'chord_group': grp, 'pitch_token_position': pitch_token_position,
                      'token_span': (span_start, i)})
        if not is_chord and not is_grace and duration is not None:
            current_onset += duration
    return notes, warnings

print('parse_lmx_notes defined')


parse_lmx_notes defined


In [5]:
_SEMITONES = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}

def _pitch_to_cp_idx(tok):
    midi = (int(tok[1]) + 1) * 12 + _SEMITONES[tok[0]]
    idx  = midi - 22
    return idx if 0 <= idx <= 85 else None

def lmx_to_cp_example(ex, pitch_classes):
    """
    Convert one JSONL example to (cp_ids, error_note_idx, target_class).
    cp_ids: list[list[int]] of shape [n_notes, 4] = [bar, pos, pitch, dur].
    Error note pitch is replaced with PITCH_MASK (87).
    Returns None only if error note position not found (0 in practice).
    """
    target = ex['target_token']
    if target not in pitch_classes:
        return None

    notes, _ = parse_lmx_notes(ex['input_tokens'], ex.get('sample_id', ''))
    err_pos   = ex['error_position']
    err_idx   = next((i for i, n in enumerate(notes)
                      if n['pitch_token_position'] == err_pos), None)
    if err_idx is None:
        return None

    cp_ids    = []
    prev_meas = -1
    for i, note in enumerate(notes):
        bar   = 0 if note['measure'] != prev_meas else 1
        prev_meas = note['measure']
        pos   = min(15, int(float(note['onset']) * 16))
        if i == err_idx:
            pitch = PITCH_MASK
        elif note['pitch'] and PITCH_RE.match(note['pitch']):
            p = _pitch_to_cp_idx(note['pitch'])
            pitch = p if p is not None else PITCH_PAD
        else:
            pitch = PITCH_PAD   # rest
        if note['duration'] is not None:
            dur = min(63, int(float(note['duration']) * 32))
        else:
            dur = DUR_PAD
        cp_ids.append([bar, pos, pitch, dur])

    return cp_ids, err_idx, pitch_classes[target]


In [6]:
def build_cp_examples(raw_list, pitch_classes, split_name):
    examples, skipped = [], 0
    for ex in raw_list:
        res = lmx_to_cp_example(ex, pitch_classes)
        if res is None:
            skipped += 1; continue
        cp_ids, err_idx, tgt = res
        examples.append({'cp_ids': cp_ids, 'error_note_idx': err_idx,
                         'target_class': tgt, 'sample_id': ex['sample_id']})
    lens = [len(e['cp_ids']) for e in examples]
    print(f'{split_name}: {len(examples)} ok, {skipped} skipped  '
          f'seq_len min={min(lens)} max={max(lens)} mean={sum(lens)/len(lens):.1f}')
    return examples

print('Building CP examples...')
t0 = time.time()
train_cp = build_cp_examples(train_raw, pitch_classes, 'train')
dev_cp   = build_cp_examples(dev_raw,   pitch_classes, 'dev')
print(f'Done in {time.time()-t0:.1f}s')


Building CP examples...
train: 9998 ok, 0 skipped  seq_len min=9 max=219 mean=66.4
dev: 499 ok, 0 skipped  seq_len min=16 max=180 mean=63.8
Done in 1.9s


## M2BERT Model

- **Backbone** (`MidiModernBERT`): 4 × Embedding(vocab_size, 256) → Linear(1024, 768)
  → ModernBERT-base (12 layers, 768 hidden). Loaded from `model_best.ckpt` with `strict=True`.
- **Head**: Dropout(0.1) → Linear(768, 52). Takes hidden state at the masked note position.
- **Frozen**: all backbone except layers 10, 11 (last 2 transformer blocks).


In [7]:
class LUT(nn.Module):
    """Lookup-table wrapper matching M2BERT checkpoint key `word_emb.N.lut.weight`."""
    def __init__(self, n, d, pad=None):
        super().__init__()
        self.lut = nn.Embedding(n, d, padding_idx=pad)
    def forward(self, x):
        return self.lut(x)


class MidiModernBERT(nn.Module):
    """M2BERT backbone: CP embeddings → ModernBERT-12L."""
    def __init__(self):
        super().__init__()
        self.word_emb = nn.ModuleList([
            LUT(4,  256, pad=BAR_PAD),    # Bar
            LUT(18, 256, pad=POS_PAD),    # Position
            LUT(88, 256, pad=PITCH_PAD),  # Pitch
            LUT(66, 256, pad=DUR_PAD),    # Duration
        ])
        self.in_linear = nn.Linear(1024, 768)
        _cfg = ModernBertConfig(
            vocab_size=10, hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=1152,
            max_position_embeddings=4096, pad_token_id=0,
        )
        self.note_bert1 = ModernBertModel(_cfg)

    def forward(self, cp_ids, attention_mask=None):
        # cp_ids: [B, S, 4]
        embs = torch.cat([self.word_emb[i](cp_ids[:, :, i]) for i in range(4)], dim=-1)
        embs = self.in_linear(embs)           # [B, S, 768]
        out  = self.note_bert1(inputs_embeds=embs, attention_mask=attention_mask)
        return out.last_hidden_state          # [B, S, 768]


class M2BERTPitchCorrection(nn.Module):
    """M2BERT backbone + classification head for pitch correction."""
    def __init__(self, backbone, n_classes):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_classes))

    def forward(self, cp_ids, error_note_idx, attention_mask=None):
        hidden  = self.backbone(cp_ids, attention_mask)                   # [B, S, 768]
        err_h   = hidden[torch.arange(hidden.size(0)), error_note_idx]    # [B, 768]
        return self.classifier(err_h)                                     # [B, N_CLASSES]


In [8]:
backbone = MidiModernBERT()
_ckpt    = torch.load(PRETRAINED / 'model_best.ckpt', map_location='cpu', weights_only=False)
backbone.load_state_dict(_ckpt['state_dict'], strict=True)
print(f'Backbone: strict=True load OK  '
      f'(pretrained epoch {_ckpt["epoch"]}, best_acc={_ckpt["best_acc"]:.4f})')

model = M2BERTPitchCorrection(backbone, N_CLASSES)
model.to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
print(f'Total params: {total_p:,}  N_CLASSES: {N_CLASSES}')


[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 9), got 50282. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 9), got 50281. This may result in unexpected behavior.
[transformers] Model config: cls_token_id must be `None` or an integer within the vocabulary (between 0 and 9), got 50281. This may result in unexpected behavior.
[transformers] Model config: sep_token_id must be `None` or an integer within the vocabulary (between 0 and 9), got 50282. This may result in unexpected behavior.


Backbone: strict=True load OK  (pretrained epoch 25, best_acc=0.9362)
Total params: 61,061,172  N_CLASSES: 52


In [9]:
# Freeze all backbone parameters
for p in model.parameters():
    p.requires_grad_(False)

# Unfreeze last 2 transformer layers (10 & 11) and classifier head
for layer_idx in (10, 11):
    for p in model.backbone.note_bert1.layers[layer_idx].parameters():
        p.requires_grad_(True)
for p in model.classifier.parameters():
    p.requires_grad_(True)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frz   = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Trainable: {n_train:,}  Frozen: {n_frz:,}')

optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
loss_fn   = nn.CrossEntropyLoss()


Trainable: 10,070,068  Frozen: 50,991,104


In [10]:
class CPDataset(Dataset):
    def __init__(self, examples): self.examples = examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

_pad_t = torch.tensor(CP_PAD, dtype=torch.long)

def collate_cp(batch):
    max_len    = max(len(e['cp_ids']) for e in batch)
    B          = len(batch)
    cp_ids_t   = _pad_t.unsqueeze(0).expand(max_len, -1).unsqueeze(0).expand(B, -1, -1).clone()
    attn_mask  = torch.zeros(B, max_len, dtype=torch.long)
    err_idx_t  = torch.zeros(B, dtype=torch.long)
    tgt_cls_t  = torch.zeros(B, dtype=torch.long)
    for i, e in enumerate(batch):
        L = len(e['cp_ids'])
        cp_ids_t[i, :L] = torch.tensor(e['cp_ids'], dtype=torch.long)
        attn_mask[i, :L] = 1
        err_idx_t[i]     = e['error_note_idx']
        tgt_cls_t[i]     = e['target_class']
    return cp_ids_t, err_idx_t, tgt_cls_t, attn_mask

# Benchmark single batch forward pass
_bmark_loader = DataLoader(CPDataset(train_cp[:64]), batch_size=64,
                            shuffle=False, collate_fn=collate_cp)
_cp, _ei, _tc, _am = next(iter(_bmark_loader))
model.eval()
with torch.no_grad():
    t0 = time.time()
    _ = model(_cp.to(DEVICE), _ei.to(DEVICE), _am.to(DEVICE))
    torch.mps.synchronize() if DEVICE.type == 'mps' else None
    print(f'Single batch (B=64): {(time.time()-t0)*1000:.1f}ms  '
          f'seq_len≈{_am.sum(1).float().mean().item():.0f}')
del _bmark_loader, _cp, _ei, _tc, _am


Single batch (B=64): 368.1ms  seq_len≈62


## Training ⚠️ TRAINS

Budget: ≤ 3 epochs **or** ≤ 10 minutes. Best checkpoint saved by dev top-1.


In [11]:
# Continue M2BERT fine-tuning up to TOTAL epoch 6.
# Requires the previous model/data/optimizer setup cells.

import json
import time
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

MAX_EPOCHS = 6
MAX_SECONDS = 180
PATIENCE = 2
BATCH_SIZE = 32

# ---------- Restore the previous best ----------

source_dir = Path(M2_RUN_DIR)
source_path = source_dir / "best_checkpoint.pt"
ckpt = torch.load(source_path, map_location="cpu", weights_only=True)

history = list(ckpt.get("metrics", []))

if "epoch" in ckpt:
    start_epoch = int(ckpt["epoch"])
elif history:
    start_epoch = int(
        max(history, key=lambda r: r["dev_top1"])["epoch"]
    )
else:
    raise ValueError(
        "Checkpoint has no epoch information; identify its epoch first."
    )

if start_epoch >= MAX_EPOCHS:
    raise ValueError(
        f"Checkpoint is already at epoch {start_epoch}; "
        "no additional training needed."
    )

model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(DEVICE)

if "optimizer_state_dict" in ckpt:
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    print("Restored model and optimizer.")
else:
    optimizer.state.clear()
    print("Restored model; optimizer state initialized fresh.")

best_dev_top1 = float(ckpt["best_dev_top1"])
best_epoch = start_epoch
metrics = [
    dict(r) for r in history
    if int(r["epoch"]) <= start_epoch
]
del ckpt

# New directory: preserve all original artifacts.
run_dir = source_dir.parent / (
    source_dir.name
    + "_resume_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
run_dir.mkdir(parents=True, exist_ok=False)
best_path = run_dir / "best_checkpoint.pt"

train_loader = DataLoader(
    CPDataset(train_cp),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_cp,
    num_workers=0,
)

dev_loader = DataLoader(
    CPDataset(dev_cp),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_cp,
    num_workers=0,
)

# ---------- Evaluation and checkpoint helpers ----------

def eval_metrics(model, loader):
    model.eval()
    n, top1, top5, ce_sum = 0, 0, 0, 0.0

    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = [
                x.to(DEVICE)
                for x in (cp_ids, err_idx, tgt, attn)
            ]

            logits = model(cp_ids, err_idx, attn)

            top1 += (logits.argmax(1) == tgt).sum().item()

            candidates = logits.topk(
                min(5, logits.size(1)), dim=1
            ).indices
            top5 += (
                candidates == tgt[:, None]
            ).any(dim=1).sum().item()

            ce_sum += F.cross_entropy(
                logits, tgt, reduction="sum"
            ).item()
            n += tgt.size(0)

    return {
        "n": n,
        "correct_top1": top1,
        "correct_top5": top5,
        "top1": top1 / n,
        "top5": top5 / n,
        "ce": ce_sum / n,
    }


def save_best(epoch, epoch_complete=True):
    # Save immediately so later training cannot change this snapshot.
    torch.save(
        {
            "epoch": epoch,
            "epoch_complete": epoch_complete,
            "model_state_dict": {
                k: v.detach().cpu()
                for k, v in model.state_dict().items()
            },
            "optimizer_state_dict": optimizer.state_dict(),
            "best_dev_top1": best_dev_top1,
            "metrics": list(metrics),
            "n_classes": N_CLASSES,
            "pitch_classes": pitch_classes,
            "source_checkpoint": str(source_path),
        },
        best_path,
    )


# The old best remains the baseline, even if continuation gets worse.
save_best(start_epoch)

print(
    f"Starting at epoch {start_epoch}; "
    f"best dev top-1={best_dev_top1:.2%}"
)
print(f"Output: {run_dir}")

# ---------- Continue training ----------

bad_epochs = 0
budget_hit = False
t_start = time.monotonic()

for epoch in range(start_epoch + 1, MAX_EPOCHS + 1):
    if time.monotonic() - t_start >= MAX_SECONDS:
        print("Training budget reached.")
        break

    model.train()
    loss_sum, n_train = 0.0, 0

    for cp_ids, err_idx, tgt, attn in train_loader:
        if time.monotonic() - t_start >= MAX_SECONDS:
            budget_hit = True
            break

        cp_ids, err_idx, tgt, attn = [
            x.to(DEVICE)
            for x in (cp_ids, err_idx, tgt, attn)
        ]

        optimizer.zero_grad(set_to_none=True)
        logits = model(cp_ids, err_idx, attn)
        loss = loss_fn(logits, tgt)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * tgt.size(0)
        n_train += tgt.size(0)

    if n_train == 0:
        break

    epoch_complete = n_train == len(train_loader.dataset)
    dev = eval_metrics(model, dev_loader)
    improved = dev["top1"] > best_dev_top1

    row = {
        "epoch": epoch,
        "epoch_complete": epoch_complete,
        "train_examples_seen": n_train,
        "train_loss": loss_sum / n_train,
        "dev_top1": dev["top1"],
        "dev_top5": dev["top5"],
        "dev_ce": dev["ce"],
        "correct_top1": dev["correct_top1"],
        "correct_top5": dev["correct_top5"],
        "dev_n": dev["n"],
        "elapsed_s": time.monotonic() - t_start,
    }
    metrics.append(row)

    if improved:
        best_dev_top1 = dev["top1"]
        best_epoch = epoch
        bad_epochs = 0
        save_best(epoch, epoch_complete)
    else:
        bad_epochs += 1

    with open(run_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    status = "complete" if epoch_complete else "partial"
    print(
        f"Epoch {epoch} ({status}): "
        f"loss={row['train_loss']:.4f} "
        f"dev={dev['correct_top1']}/{dev['n']} "
        f"({dev['top1']:.2%}) "
        f"top-5={dev['top5']:.2%}"
        + ("  *best*" if improved else "")
    )

    if budget_hit or time.monotonic() - t_start >= MAX_SECONDS:
        print("Training budget reached; stopping.")
        break

    if bad_epochs >= PATIENCE:
        print(
            f"Early stopping: {PATIENCE} epochs "
            "without improvement."
        )
        break

# ---------- Restore and evaluate the selected best ----------

selected = torch.load(
    best_path, map_location="cpu", weights_only=True
)
model.load_state_dict(selected["model_state_dict"], strict=True)
optimizer.load_state_dict(selected["optimizer_state_dict"])
del selected
model.eval()

final = eval_metrics(model, dev_loader)

with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open(run_dir / "final_dev_metrics.json", "w") as f:
    json.dump({"best_epoch": best_epoch, **final}, f, indent=2)

# Following cells use this run; the original directory stays intact.
M2_RUN_DIR = run_dir

print(f"\nBest epoch: {best_epoch}")
print(
    f"Top-1: {final['correct_top1']}/{final['n']} "
    f"= {final['top1']:.2%}"
)
print(
    f"Top-5: {final['correct_top5']}/{final['n']} "
    f"= {final['top5']:.2%}"
)
print(f"Saved to: {M2_RUN_DIR}")

Restored model; optimizer state initialized fresh.
Starting at epoch 3; best dev top-1=60.52%
Output: /Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml/data/runs/m2bert_finetune_resume_20261030_183302_306020
Epoch 4 (complete): loss=0.1138 dev=301/499 (60.32%) top-5=83.97%
Epoch 5 (complete): loss=0.0222 dev=312/499 (62.53%) top-5=85.57%  *best*
Epoch 6 (complete): loss=0.0422 dev=286/499 (57.31%) top-5=81.56%

Best epoch: 5
Top-1: 312/499 = 62.53%
Top-5: 427/499 = 85.57%
Saved to: /Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml/data/runs/m2bert_finetune_resume_20261030_183302_306020


## Results — M2BERT vs v5 📊

Load best checkpoint, evaluate on all 499 dev examples, compare with v5.


In [12]:
# 📊 Evaluate M2BERT on dev + compare with v5.
# Requires: best_checkpoint.pt saved by the TRAINS cell.

# Load best checkpoint
_saved = torch.load(M2_RUN_DIR / 'best_checkpoint.pt', map_location='cpu', weights_only=False)
model.load_state_dict(_saved['model_state_dict'])
model.to(DEVICE)
model.eval()
print(f'Loaded checkpoint: best_dev_top1={_saved["best_dev_top1"]:.4f}')

# Full dev evaluation with per-example predictions
all_preds = []
with torch.inference_mode():
    for cp_ids, err_idx, tgt, attn in dev_loader:
        logits   = model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
        probs    = logits.softmax(dim=-1).cpu()
        top5_ids = probs.topk(5, dim=-1).indices.tolist()
        for b in range(tgt.size(0)):
            pred_id  = top5_ids[b][0]
            tgt_id   = tgt[b].item()
            all_preds.append({
                'sample_id':      dev_cp[len(all_preds)]['sample_id'],
                'target_token':   id2pitch[tgt_id],
                'predicted_token': id2pitch[pred_id],
                'correct':        (pred_id == tgt_id),
                'top5_candidates': [id2pitch[x] for x in top5_ids[b]],
            })

with open(M2_RUN_DIR / 'dev_predictions.json', 'w') as f:
    json.dump(all_preds, f, indent=2)

m2_t1 = sum(1 for p in all_preds if p['correct'])
m2_t5 = sum(1 for p in all_preds if p['target_token'] in p['top5_candidates'])
n_dev = len(all_preds)
print(f'M2BERT  dev: top-1 = {m2_t1}/{n_dev} ({m2_t1/n_dev:.2%})  '
      f'top-5 = {m2_t5}/{n_dev} ({m2_t5/n_dev:.2%})')

# ── Load v5 for comparison ─────────────────────────────────────
V5_DIR     = ML_DIR / 'data' / 'runs' / 'v5_voice_conditioned'
_v5_preds  = json.load(open(V5_DIR / 'dev_predictions.json'))
v5_t1 = sum(1 for p in _v5_preds if p['correct'])
v5_t5 = sum(1 for p in _v5_preds if p['target_token'] in p['top5_candidates'])
print(f'v5      dev: top-1 = {v5_t1}/{n_dev} ({v5_t1/n_dev:.2%})  '
      f'top-5 = {v5_t5}/{n_dev} ({v5_t5/n_dev:.2%})')

# ── Venn: overlap ──────────────────────────────────────────────
m2_ids = {p['sample_id'] for p in all_preds if p['correct']}
v5_ids = {p['sample_id'] for p in _v5_preds if p['correct']}
both  = len(m2_ids & v5_ids)
only_m2 = len(m2_ids - v5_ids)
only_v5 = len(v5_ids - m2_ids)
print(f'\nVenn overlap  both={both}  only_M2BERT={only_m2}  only_v5={only_v5}')

# ── Summary table ─────────────────────────────────────────────
_met = json.load(open(M2_RUN_DIR / 'metrics.json'))
n_ep = len(_met)
partial = ' (partial)' if n_ep < 3 else ''
print(f'\n{"Model":<22} {"Top-1":>10} {"Top-5":>10} {"Epochs":>8}')
print('─' * 55)
print(f'{"v5 (voice-conditioned)":<22} '
      f'{v5_t1}/{n_dev} ({v5_t1/n_dev:.2%}) {" ":>4} '
      f'{v5_t5}/{n_dev} ({v5_t5/n_dev:.2%}) '
      f'{"10":>8}')
print(f'{"M2BERT fine-tuned":<22} '
      f'{m2_t1}/{n_dev} ({m2_t1/n_dev:.2%}) {" ":>4} '
      f'{m2_t5}/{n_dev} ({m2_t5/n_dev:.2%}) '
      f'{str(n_ep)+partial:>8}')


Loaded checkpoint: best_dev_top1=0.6253
M2BERT  dev: top-1 = 312/499 (62.53%)  top-5 = 427/499 (85.57%)
v5      dev: top-1 = 83/499 (16.63%)  top-5 = 261/499 (52.30%)

Venn overlap  both=58  only_M2BERT=254  only_v5=25

Model                       Top-1      Top-5   Epochs
───────────────────────────────────────────────────────
v5 (voice-conditioned) 83/499 (16.63%)      261/499 (52.30%)       10
M2BERT fine-tuned      312/499 (62.53%)      427/499 (85.57%)        6


## Extended Training (epochs 5–20) ⚠️ TRAINS

Resume from epoch 4 best checkpoint (304/499 = 60.92% top-1, 417/499 = 83.57% top-5).

| Parameter | Value |
|-----------|-------|
| Source checkpoint | `m2bert_finetune_resume_20261028_101120_113800/best_checkpoint.pt` (epoch 4) |
| Resume from | epoch 5 |
| Max epochs | 20 |
| Early stopping | patience = 4 (no dev top-1 improvement) |
| Budget | 600 s |
| LR / batch / freeze | unchanged (1e-4, 32, layers 10–11 + head) |

**⚠️ Do not run the next cell until the machine is confirmed safe.**


In [13]:
# ⚠️ TRAINS — Extended fine-tuning (epochs 5–20, patience=4, 600 s budget).
# Requires: all setup cells (1–12) executed first.
# Loads epoch-4 best checkpoint; saves to a new run directory.

from datetime import datetime
import torch.nn.functional as F

# ── Configuration ──────────────────────────────────────────────
RESUME_DIR   = ML_DIR / "data" / "runs" / "m2bert_finetune_resume_20261028_101120_113800"
START_EPOCH  = 4          # checkpoint epoch (training resumes at START_EPOCH + 1)
MAX_EPOCH    = 20
PATIENCE     = 4          # early stopping on dev top-1
MAX_SECONDS  = 600
BATCH_SIZE   = 32

# ── Load epoch-4 checkpoint ───────────────────────────────────
_ckpt = torch.load(RESUME_DIR / "best_checkpoint.pt", map_location="cpu", weights_only=False)
model.load_state_dict(_ckpt["model_state_dict"], strict=True)
model.to(DEVICE)
optimizer.load_state_dict(_ckpt["optimizer_state_dict"])
baseline_top1 = _ckpt["best_dev_top1"]          # 0.6092 = 304/499
print(f"Loaded epoch {START_EPOCH}: dev top-1 = {baseline_top1:.4f}")
del _ckpt

# ── Output directory (preserves original) ─────────────────────
EXT_RUN_DIR = ML_DIR / "data" / "runs" / (
    "m2bert_finetune_extended_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
EXT_RUN_DIR.mkdir(parents=True, exist_ok=False)
print(f"Output: {EXT_RUN_DIR}")

# ── DataLoaders ───────────────────────────────────────────────
train_loader = DataLoader(CPDataset(train_cp), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_cp, num_workers=0)
dev_loader   = DataLoader(CPDataset(dev_cp), batch_size=64,
                          shuffle=False, collate_fn=collate_cp, num_workers=0)

# ── Evaluation helper ─────────────────────────────────────────
def _eval_dev(model, loader, device):
    model.eval()
    n = top1 = top5 = 0; ce_sum = 0.0
    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = (
                cp_ids.to(device), err_idx.to(device),
                tgt.to(device), attn.to(device)
            )
            logits = model(cp_ids, err_idx, attn)
            top1 += (logits.argmax(1) == tgt).sum().item()
            top5 += (logits.topk(min(5, logits.size(1)), dim=1)
                      .indices == tgt[:, None]).any(1).sum().item()
            ce_sum += F.cross_entropy(logits, tgt, reduction="sum").item()
            n += tgt.size(0)
    return {"n": n, "top1": top1/n, "top5": top5/n, "ce": ce_sum/n,
            "correct_top1": top1, "correct_top5": top5}

# ── Training loop ─────────────────────────────────────────────
metrics      = []
best_top1    = baseline_top1
best_epoch   = START_EPOCH
best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
best_optim   = optimizer.state_dict()
bad_epochs   = 0
t_start      = time.monotonic()

for epoch in range(START_EPOCH + 1, MAX_EPOCH + 1):
    elapsed = time.monotonic() - t_start
    if elapsed >= MAX_SECONDS:
        print(f"Budget reached before epoch {epoch} ({elapsed:.0f}s)"); break

    model.train()
    loss_sum, n_seen = 0.0, 0
    for cp_ids, err_idx, tgt, attn in train_loader:
        if time.monotonic() - t_start >= MAX_SECONDS:
            break
        cp_ids, err_idx, tgt, attn = (
            cp_ids.to(DEVICE), err_idx.to(DEVICE),
            tgt.to(DEVICE), attn.to(DEVICE)
        )
        optimizer.zero_grad(set_to_none=True)
        logits = model(cp_ids, err_idx, attn)
        loss = loss_fn(logits, tgt)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * tgt.size(0)
        n_seen += tgt.size(0)

    if n_seen == 0:
        break
    complete = (n_seen == len(train_cp))

    dev = _eval_dev(model, dev_loader, DEVICE)
    improved = dev["top1"] > best_top1

    row = {"epoch": epoch, "complete": complete, "n_seen": n_seen,
           "train_loss": loss_sum / n_seen,
           "dev_top1": dev["top1"], "dev_top5": dev["top5"], "dev_ce": dev["ce"],
           "correct_top1": dev["correct_top1"], "correct_top5": dev["correct_top5"],
           "dev_n": dev["n"], "elapsed_s": round(time.monotonic() - t_start, 1)}
    metrics.append(row)

    star = "  ← best" if improved else ""
    tag  = "" if complete else " (partial)"
    print(f"Epoch {epoch}{tag}: loss={row['train_loss']:.4f}  "
          f"dev={dev['correct_top1']}/{dev['n']} ({dev['top1']:.2%})  "
          f"top-5={dev['top5']:.2%}  ce={dev['ce']:.4f}  "
          f"t={row['elapsed_s']:.0f}s{star}")

    if improved:
        best_top1  = dev["top1"]
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_optim = optimizer.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1

    # Save metrics after every epoch
    with open(EXT_RUN_DIR / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    if bad_epochs >= PATIENCE:
        print(f"Early stopping: {PATIENCE} epochs without improvement."); break
    if time.monotonic() - t_start >= MAX_SECONDS:
        print("Budget reached; stopping."); break

# ── Save best checkpoint ─────────────────────────────────────
torch.save(
    {"epoch": best_epoch, "model_state_dict": best_state,
     "optimizer_state_dict": best_optim, "best_dev_top1": best_top1,
     "metrics": metrics, "n_classes": N_CLASSES,
     "pitch_classes": pitch_classes,
     "source_checkpoint": str(RESUME_DIR / "best_checkpoint.pt")},
    EXT_RUN_DIR / "best_checkpoint.pt",
)

# ── Load best and report ──────────────────────────────────────
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()
final = _eval_dev(model, dev_loader, DEVICE)

with open(EXT_RUN_DIR / "final_dev_metrics.json", "w") as f:
    json.dump({"best_epoch": best_epoch, **final}, f, indent=2)

# ── Print summary ─────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Best epoch: {best_epoch}")
print(f"  Top-1: {final['correct_top1']}/{final['n']} = {final['top1']:.2%}")
print(f"  Top-5: {final['correct_top5']}/{final['n']} = {final['top5']:.2%}")
print(f"  CE:    {final['ce']:.4f}")
print(f"\nBaseline (epoch 4): 304/499 = 60.92%  top-5 = 83.57%")
dt1 = final["correct_top1"] - 304
dt5 = final["correct_top5"] - 417
print(f"Δ top-1: {dt1:+d} examples ({(final['top1'] - 0.6092)*100:+.2f}pp)")
print(f"Δ top-5: {dt5:+d} examples ({(final['top5'] - 0.8357)*100:+.2f}pp)")
print(f"\nSaved to: {EXT_RUN_DIR}")

# Point M2_RUN_DIR to the new run for the results cell below
M2_RUN_DIR = EXT_RUN_DIR


Loaded epoch 4: dev top-1 = 0.6092
Output: /Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml/data/runs/m2bert_finetune_extended_20261030_183422
Epoch 5: loss=0.0264  dev=303/499 (60.72%)  top-5=85.17%  ce=1.8581  t=26s
Epoch 6: loss=0.0355  dev=293/499 (58.72%)  top-5=82.16%  ce=2.0608  t=51s
Epoch 7: loss=0.0340  dev=292/499 (58.52%)  top-5=84.77%  ce=2.1521  t=77s
Epoch 8: loss=0.0329  dev=291/499 (58.32%)  top-5=81.16%  ce=2.3192  t=103s
Early stopping: 4 epochs without improvement.

────────────────────────────────────────────────────────────
Best epoch: 4
  Top-1: 304/499 = 60.92%
  Top-5: 417/499 = 83.57%
  CE:    1.8498

Baseline (epoch 4): 304/499 = 60.92%  top-5 = 83.57%
Δ top-1: +0 examples (+0.00pp)
Δ top-5: +0 examples (-0.00pp)

Saved to: /Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml/data/runs/m2bert_finetune_extended_20261030_183422


## Demo — Pitch Prediction → Corrected MusicXML → MIDI

Select 6 dev samples (seed=42), run inference with the epoch-5 checkpoint
(312/499 = 62.53% top-1), then generate three MusicXML + MIDI versions per sample:

| Version | Description |
|---------|-------------|
| **ground-truth** | Original score (correct pitch) |
| **corrupted** | Score with the synthetic error (replacement pitch) |
| **model-corrected** | Score with the model's top-1 prediction applied |


In [ ]:
# Select 6 dev samples (seed=42) and run inference.
# Requires: cells 1–12 (setup, model classes, data), cell 18 (extended training) OR
# manual checkpoint load below.

import copy
from lxml import etree
import mido

DEMO_CKPT = ML_DIR / "data" / "runs" / "m2bert_finetune_resume_20261030_183302_306020" / "best_checkpoint.pt"
DEMO_DIR  = ML_DIR / "demo"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

# ── Load checkpoint ────────────────────────────────────────
_ckpt = torch.load(DEMO_CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(_ckpt["model_state_dict"], strict=True)
model.to(DEVICE)
model.eval()
print(f"Loaded checkpoint: dev top-1 = {_ckpt['best_dev_top1']:.4f}")
del _ckpt

# ── Select 6 samples ───────────────────────────────────────
rng = random.Random(42)
DEMO_INDICES = sorted(rng.sample(range(len(dev_cp)), 6))
demo_examples = [dev_cp[i] for i in DEMO_INDICES]
demo_raw      = [dev_raw[i] for i in DEMO_INDICES]

with open(DEMO_DIR / "demo_sample_ids.json", "w") as f:
    json.dump([{"index": i, "sample_id": dev_raw[i]["sample_id"]} for i in DEMO_INDICES], f, indent=2)

# ── Batch inference ────────────────────────────────────────
demo_loader = DataLoader(CPDataset(demo_examples), batch_size=6,
                         shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(demo_loader))

with torch.inference_mode():
    logits = model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
    probs  = logits.softmax(dim=-1).cpu()
    top5   = probs.topk(5, dim=-1)

# ── Display results ────────────────────────────────────────
demo_results = []
print(f"{'Idx':>4} {'Sample ID':>30}  {'Error':>5} {'Truth':>5} {'Pred':>5} {'OK?':>4}  Top-5")
print("─" * 95)
for b in range(6):
    raw       = demo_raw[b]
    tgt_id    = tgt[b].item()
    pred_id   = top5.indices[b][0].item()
    correct   = (pred_id == tgt_id)
    top5_list = [id2pitch[top5.indices[b][k].item()] for k in range(5)]
    res = {
        "dev_index":    DEMO_INDICES[b],
        "sample_id":    raw["sample_id"],
        "error_pitch":  raw["replacement_token"],
        "truth_pitch":  raw["target_token"],
        "pred_pitch":   id2pitch[pred_id],
        "correct":      correct,
        "top5":         top5_list,
    }
    demo_results.append(res)
    mark = "  ✓" if correct else "  ✗"
    print(f"{DEMO_INDICES[b]:4d} {raw['sample_id']:>30}  "
          f"{raw['replacement_token']:>5} {raw['target_token']:>5} "
          f"{id2pitch[pred_id]:>5}{mark}  {top5_list}")


In [ ]:
# MusicXML pitch editing and MIDI export utilities.

_STEP_SEMI = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}

def _xml_tag(name, ns):
    return f"{{{ns}}}{name}" if ns else name


def find_error_note_index(ex, parse_lmx_notes):
    """Return the note index (in parse_lmx_notes order = XML note order)
    of the corrupted pitch in a JSONL example."""
    notes, _ = parse_lmx_notes(ex["input_tokens"], ex.get("sample_id", ""))
    return next(i for i, n in enumerate(notes)
               if n["pitch_token_position"] == ex["error_position"])


def set_note_pitch(note_el, pitch_token, ns=""):
    """Rewrite <step>/<octave> of a <note> element; remove <alter>/<accidental>."""
    t = lambda name: _xml_tag(name, ns)
    pitch_el = note_el.find(t("pitch"))
    if pitch_el is None:
        return
    pitch_el.find(t("step")).text   = pitch_token[0]
    pitch_el.find(t("octave")).text = pitch_token[1:]
    alter = pitch_el.find(t("alter"))
    if alter is not None:
        pitch_el.remove(alter)
    acc = note_el.find(t("accidental"))
    if acc is not None:
        note_el.remove(acc)


def make_edited_xml(xml_path, note_index, new_pitch):
    """Return a deep-copied etree with one note's pitch changed."""
    tree = etree.parse(str(xml_path))
    root = tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    notes = root.findall(f".//{t('note')}")
    set_note_pitch(notes[note_index], new_pitch, ns)
    return tree


def musicxml_to_midi(xml_tree, midi_path):
    """Minimal MusicXML→MIDI converter using lxml + mido."""
    root = xml_tree.getroot()
    ns   = root.nsmap.get(None, "")
    t    = lambda name: _xml_tag(name, ns)
    TPQN = 480
    divisions = 1
    events = []       # (onset_tick, midi_note, dur_tick)
    current_time = 0  # in XML divisions
    prev_onset   = 0

    for measure in root.findall(f".//{t('measure')}"):
        for attr in measure.findall(t("attributes")):
            d = attr.find(t("divisions"))
            if d is not None:
                divisions = int(d.text)
        scale = TPQN / divisions
        for el in measure:
            ltag = el.tag.split("}")[-1] if "}" in el.tag else el.tag
            if ltag == "note":
                chord_el = el.find(t("chord"))
                pitch_el = el.find(t("pitch"))
                dur_el   = el.find(t("duration"))
                dur = int(dur_el.text) if dur_el is not None else 0
                onset = prev_onset if chord_el is not None else current_time
                if chord_el is None:
                    prev_onset = current_time
                if pitch_el is not None:
                    step  = pitch_el.find(t("step")).text
                    octave = int(pitch_el.find(t("octave")).text)
                    alt_el = pitch_el.find(t("alter"))
                    alter  = int(float(alt_el.text)) if alt_el is not None else 0
                    midi_n = (octave + 1) * 12 + _STEP_SEMI[step] + alter
                    events.append((int(onset * scale), midi_n, max(1, int(dur * scale))))
                if chord_el is None:
                    current_time += dur
            elif ltag == "backup":
                current_time -= int(el.find(t("duration")).text)
            elif ltag == "forward":
                current_time += int(el.find(t("duration")).text)

    mid   = mido.MidiFile(ticks_per_beat=TPQN)
    track = mido.MidiTrack()
    mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=mido.bpm2tempo(120)))
    msgs = []
    for onset, pitch, dur in events:
        msgs.append((onset, 1, pitch, 80))
        msgs.append((onset + dur, 0, pitch, 0))
    msgs.sort(key=lambda m: (m[0], m[1]))
    prev_tick = 0
    for tick, on, pitch, vel in msgs:
        delta = tick - prev_tick
        mtype = "note_on" if on else "note_off"
        track.append(mido.Message(mtype, note=pitch, velocity=vel, time=delta))
        prev_tick = tick
    mid.save(str(midi_path))
    return len(events)

print("MusicXML/MIDI helpers defined")


In [ ]:
# Generate ground-truth / corrupted / model-corrected MusicXML + MIDI
# for each of the 6 demo samples.

from shutil import copyfile

def _xml_path(sample_id):
    parts = sample_id.split("/")
    return DATASET_ROOT / "samples" / parts[1] / (parts[2] + ".musicxml")

summary_rows = []

for b, res in enumerate(demo_results):
    raw  = demo_raw[b]
    sid  = res["sample_id"]
    slug = sid.replace("/", "_")       # e.g. samples_6404150_p8-s2
    sdir = DEMO_DIR / slug
    sdir.mkdir(exist_ok=True)

    xml_src = _xml_path(sid)
    note_idx = find_error_note_index(raw, parse_lmx_notes)

    # ── 1. Ground truth: copy original ───────────────────────
    gt_xml = sdir / "ground_truth.musicxml"
    copyfile(xml_src, gt_xml)
    gt_tree = etree.parse(str(gt_xml))
    gt_midi = sdir / "ground_truth.mid"
    musicxml_to_midi(gt_tree, gt_midi)

    # ── 2. Corrupted: apply replacement pitch ────────────────
    corr_tree = make_edited_xml(xml_src, note_idx, res["error_pitch"])
    corr_xml  = sdir / "corrupted.musicxml"
    corr_tree.write(str(corr_xml), xml_declaration=True, encoding="UTF-8")
    corr_midi = sdir / "corrupted.mid"
    musicxml_to_midi(corr_tree, corr_midi)

    # ── 3. Model corrected: apply model prediction ───────────
    pred_tree = make_edited_xml(xml_src, note_idx, res["pred_pitch"])
    pred_xml  = sdir / "model_corrected.musicxml"
    pred_tree.write(str(pred_xml), xml_declaration=True, encoding="UTF-8")
    pred_midi = sdir / "model_corrected.mid"
    musicxml_to_midi(pred_tree, pred_midi)

    mark = "✓" if res["correct"] else "✗"
    summary_rows.append({
        "idx": res["dev_index"], "sample_id": sid,
        "error": res["error_pitch"], "truth": res["truth_pitch"],
        "pred": res["pred_pitch"], "ok": mark,
        "dir": str(sdir.relative_to(ML_DIR)),
    })
    print(f"[{b+1}/6] {slug}  truth={res['truth_pitch']}  pred={res['pred_pitch']} {mark}")

# ── Save summary ──────────────────────────────────────────
with open(DEMO_DIR / "demo_summary.json", "w") as f:
    json.dump(summary_rows, f, indent=2)

# ── Display file listing ──────────────────────────────────
print(f"\n{'─'*80}")
print(f"{'#':>2} {'Dev':>4} {'Truth':>5} → {'Pred':>5} {'OK':>3}  Files")
print(f"{'─'*80}")
for i, r in enumerate(summary_rows):
    print(f"{i+1:2d} {r['idx']:4d} {r['truth']:>5} → {r['pred']:>5}  {r['ok']}"
          f"   {r['dir']}/{{ground_truth,corrupted,model_corrected}}.{{musicxml,mid}}")
print(f"\nAll outputs: {DEMO_DIR.relative_to(ML_DIR)}/")


## Duration Correction — Fine-tuning

Task: predict the masked duration of a single corrupted note.  
Same source samples and score-level split as pitch correction.  
One note per sample; target = exact Fraction quarterLength.

**Backbone**: M2BERT pretrained (same checkpoint as pitch, reloaded fresh).  
**MFC baseline**: most-frequent duration class from train.

In [ ]:
# ── Duration Correction: Data Construction ───────────────────────
# Build duration-corruption examples from the same source samples.
# One note per sample, deterministic selection (hash of sample_id).

import hashlib as _hl
from datetime import datetime
import torch.nn.functional as F

DUR_MASK_ID = CP_TOK2ID['Duration']['Duration <MASK>']  # 65

# Replacement duration pool: (LMX_tokens, Fraction_whole_notes)
_DUR_REPL_POOL = [
    (['whole'],          _Frac(1)),
    (['half'],           _Frac(1, 2)),
    (['quarter'],        _Frac(1, 4)),
    (['eighth'],         _Frac(1, 8)),
    (['16th'],           _Frac(1, 16)),
    (['half', 'dot'],    _Frac(3, 4)),
    (['quarter', 'dot'], _Frac(3, 8)),
    (['eighth', 'dot'],  _Frac(3, 16)),
]

def _find_dur_span(tokens, note):
    """Return (start, end) index of duration tokens within note's token_span."""
    s, e = note['token_span']
    for i in range(s, e):
        if tokens[i] in _DURATION_BASE:
            j = i + 1
            if j < e and _TIME_MOD_RE.match(tokens[j]):
                j += 1
            while j < e and tokens[j] == 'dot':
                j += 1
            return i, j
    return None, None

def _dur_token_info(tokens, note):
    """Extract (note_type, n_dots, tuplet_str|None) from LMX tokens."""
    ds, de = _find_dur_span(tokens, note)
    if ds is None:
        return None, 0, None
    ntype = tokens[ds]; j = ds + 1
    tup = None
    if j < de and _TIME_MOD_RE.match(tokens[j]):
        tup = tokens[j]; j += 1
    dots = 0
    while j < de and tokens[j] == 'dot':
        dots += 1; j += 1
    return ntype, dots, tup

def build_dur_split(sample_ids, label_map=None):
    """Build duration-correction CP examples for one split.
    
    For each sample: load clean LMX, select one pitched non-grace note,
    replace its duration tokens, re-parse to get corrupted onsets,
    convert to CP with DUR_MASK at error position.
    
    Returns (cp_list, label_map, details, stats).
    """
    build_map = (label_map is None)
    raw = []
    skip = Counter()
    note_excl = Counter()

    for sid in sample_ids:
        lmx_p = _lmx_path(sid)
        if not lmx_p.exists():
            skip['no_lmx'] += 1; continue

        clean_toks = lmx_p.read_text().split()
        notes, _ = parse_lmx_notes(clean_toks, sid)

        cands = []
        for ni, n in enumerate(notes):
            if n['is_grace']:       note_excl['grace'] += 1; continue
            if n['is_rest']:        note_excl['rest'] += 1; continue
            if n['duration'] is None or n['pitch'] is None:
                note_excl['no_dur_or_pitch'] += 1; continue
            if _find_dur_span(clean_toks, n)[0] is None:
                note_excl['dur_tokens_missing'] += 1; continue
            cands.append(ni)

        if not cands:
            skip['no_candidate'] += 1; continue

        h = int(_hl.md5(sid.encode()).hexdigest(), 16)
        sel = cands[h % len(cands)]
        note = notes[sel]
        true_dur = note['duration']           # Fraction, whole-note units
        true_ql  = str(true_dur * 4)          # quarterLength label

        rng = random.Random(h)
        pool = [(t, f) for t, f in _DUR_REPL_POOL if f != true_dur]
        rtoks, rfrac = pool[rng.randint(0, len(pool) - 1)]

        ds, de = _find_dur_span(clean_toks, note)
        corrupted = clean_toks[:ds] + rtoks + clean_toks[de:]
        ntype, dots, tup = _dur_token_info(clean_toks, note)

        raw.append({
            'sid': sid, 'corrupted': corrupted,
            'epp': note['pitch_token_position'],
            'true_ql': true_ql, 'repl_ql': str(rfrac * 4),
            'ntype': ntype, 'dots': dots, 'tup': tup,
        })

    # Label map from train distribution (most-frequent first)
    if build_map:
        ql_counts = Counter(r['true_ql'] for r in raw)
        label_map = {ql: i for i, (ql, _) in enumerate(ql_counts.most_common())}

    oov_qls = sorted({r['true_ql'] for r in raw if r['true_ql'] not in label_map})

    # CP conversion — re-parse corrupted LMX for correct onsets
    cp_list, details = [], []
    for r in raw:
        if r['true_ql'] not in label_map:
            skip['oov_dur'] += 1; continue
        cnotes, _ = parse_lmx_notes(r['corrupted'], r['sid'])
        err_idx = next((i for i, n in enumerate(cnotes)
                        if n['pitch_token_position'] == r['epp']), None)
        if err_idx is None:
            skip['err_not_found'] += 1; continue

        cp_ids = []
        prev_m = -1
        for i, n in enumerate(cnotes):
            bar = 0 if n['measure'] != prev_m else 1; prev_m = n['measure']
            pos = min(15, int(float(n['onset']) * 16))
            pitch = PITCH_PAD
            if n['pitch'] and PITCH_RE.match(n['pitch']):
                p = _pitch_to_cp_idx(n['pitch'])
                if p is not None: pitch = p
            dur = DUR_MASK_ID if i == err_idx else (
                min(63, int(float(n['duration']) * 32))
                if n['duration'] is not None else DUR_PAD)
            cp_ids.append([bar, pos, pitch, dur])

        cp_list.append({'cp_ids': cp_ids, 'error_note_idx': err_idx,
                        'target_class': label_map[r['true_ql']],
                        'sample_id': r['sid']})
        details.append({'sample_id': r['sid'], 'true_ql': r['true_ql'],
                        'repl_ql': r['repl_ql'], 'note_type': r['ntype'],
                        'dots': r['dots'], 'tuplet': r['tup']})

    stats = {'input_samples': len(sample_ids), 'examples': len(cp_list),
             'skip': dict(skip), 'note_exclusions': dict(note_excl),
             'oov_durations': oov_qls}
    return cp_list, label_map, details, stats

# ── Build ────────────────────────────────────────────────────────
t0 = time.time()

train_sids = list(dict.fromkeys(ex['sample_id'] for ex in train_raw))
dev_sids   = list(dict.fromkeys(ex['sample_id'] for ex in dev_raw))
print(f"Source samples: train={len(train_sids)}, dev={len(dev_sids)}")

dur_train_cp, dur_label_map, dur_train_details, dur_train_stats = \
    build_dur_split(train_sids)
dur_dev_cp, _, dur_dev_details, dur_dev_stats = \
    build_dur_split(dev_sids, label_map=dur_label_map)

dur_id2label = {v: k for k, v in dur_label_map.items()}
N_DUR_CLASSES = len(dur_label_map)

print(f"\nDuration classes: {N_DUR_CLASSES}")
print(f"Label map (class → QL): {dur_label_map}")
print(f"\nTrain: {dur_train_stats['examples']} examples from "
      f"{dur_train_stats['input_samples']} samples")
print(f"  Skip: {dur_train_stats['skip']}")
print(f"  Note exclusions: {dur_train_stats['note_exclusions']}")
print(f"Dev:   {dur_dev_stats['examples']} examples from "
      f"{dur_dev_stats['input_samples']} samples")
print(f"  Skip: {dur_dev_stats['skip']}")
print(f"  Note exclusions: {dur_dev_stats['note_exclusions']}")
if dur_dev_stats['oov_durations']:
    print(f"  Dev OOV durations: {dur_dev_stats['oov_durations']}")
else:
    print(f"  All dev durations seen in train.")

# MFC baseline
_train_dur_counts = Counter(d['true_ql'] for d in dur_train_details)
dur_mfc_label, dur_mfc_train_n = _train_dur_counts.most_common(1)[0]
dur_mfc_cls = dur_label_map[dur_mfc_label]
dur_mfc_hits = sum(1 for e in dur_dev_cp if e['target_class'] == dur_mfc_cls)
print(f"\nMFC: '{dur_mfc_label}' QL ({dur_mfc_train_n} in train) → "
      f"dev hits: {dur_mfc_hits}/{len(dur_dev_cp)} "
      f"({dur_mfc_hits/len(dur_dev_cp):.2%})")

# Top-10 duration distribution
print(f"\nTop-10 durations in train:")
for ql, cnt in _train_dur_counts.most_common(10):
    print(f"  {ql:>8s} QL: {cnt:>5d} ({cnt/len(dur_train_details)*100:.1f}%)")

print(f"\nData construction: {time.time()-t0:.1f}s")

In [ ]:
# ── Duration Model Setup + Round 1 Training ──────────────────────
# Fresh backbone from pretrained M2BERT, new duration classification head.
# Freeze all except layers 10-11 and head. Max 6 epochs, patience=2.

DUR_RUN_DIR = ML_DIR / 'data' / 'runs' / (
    'm2bert_dur_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
DUR_RUN_DIR.mkdir(parents=True, exist_ok=True)

# ── Fresh backbone ───────────────────────────────────────────
dur_backbone = MidiModernBERT()
_ckpt = torch.load(PRETRAINED / 'model_best.ckpt', map_location='cpu',
                    weights_only=False)
dur_backbone.load_state_dict(_ckpt['state_dict'], strict=True)
print(f'Backbone loaded (pretrained epoch {_ckpt["epoch"]})')
del _ckpt

# Reuse M2BERTPitchCorrection class — same architecture, different head size
dur_model = M2BERTPitchCorrection(dur_backbone, N_DUR_CLASSES)
dur_model.to(DEVICE)

# Freeze all, unfreeze layers 10-11 + classifier
for p in dur_model.parameters():
    p.requires_grad_(False)
for layer_idx in (10, 11):
    for p in dur_model.backbone.note_bert1.layers[layer_idx].parameters():
        p.requires_grad_(True)
for p in dur_model.classifier.parameters():
    p.requires_grad_(True)

n_train_p = sum(p.numel() for p in dur_model.parameters() if p.requires_grad)
n_frz_p   = sum(p.numel() for p in dur_model.parameters() if not p.requires_grad)
print(f'Trainable: {n_train_p:,}  Frozen: {n_frz_p:,}  '
      f'Head out: {N_DUR_CLASSES} classes')

dur_optimizer = torch.optim.Adam(
    [p for p in dur_model.parameters() if p.requires_grad], lr=1e-4)
dur_loss_fn = nn.CrossEntropyLoss()

# Save config and selection records
_dur_config = {
    'n_classes': N_DUR_CLASSES, 'lr': 1e-4, 'batch_size': 32,
    'max_epochs': 6, 'patience': 2, 'frozen_layers': '0-9',
    'trainable_layers': '10-11 + head',
    'train_examples': len(dur_train_cp), 'dev_examples': len(dur_dev_cp),
    'mfc_label': dur_mfc_label, 'mfc_dev_hits': dur_mfc_hits,
}
with open(DUR_RUN_DIR / 'config.json', 'w') as f:
    json.dump(_dur_config, f, indent=2)
with open(DUR_RUN_DIR / 'dur_label_map.json', 'w') as f:
    json.dump(dur_label_map, f, indent=2)
with open(DUR_RUN_DIR / 'train_selection.json', 'w') as f:
    json.dump(dur_train_details, f, indent=2)
with open(DUR_RUN_DIR / 'dev_selection.json', 'w') as f:
    json.dump(dur_dev_details, f, indent=2)

# ── Data loaders ─────────────────────────────────────────────
dur_train_loader = DataLoader(CPDataset(dur_train_cp), batch_size=32,
                              shuffle=True, collate_fn=collate_cp, num_workers=0)
dur_dev_loader   = DataLoader(CPDataset(dur_dev_cp), batch_size=64,
                              shuffle=False, collate_fn=collate_cp, num_workers=0)

def dur_eval(mdl, loader):
    mdl.eval()
    n = top1 = top3 = 0; ce_sum = 0.0
    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
            logits = mdl(cp_ids, err_idx, attn)
            top1 += (logits.argmax(1) == tgt).sum().item()
            top3 += (logits.topk(min(3, logits.size(1)), dim=1
                     ).indices == tgt[:, None]).any(1).sum().item()
            ce_sum += F.cross_entropy(logits, tgt, reduction='sum').item()
            n += tgt.size(0)
    return {'n': n, 'top1': top1, 'top3': top3,
            'top1_acc': top1 / n, 'top3_acc': top3 / n, 'ce': ce_sum / n}

# ── Training Round 1 ────────────────────────────────────────
MAX_SECONDS_TOTAL = 600
dur_metrics_r1 = []
best_dur_top1 = 0.0
best_dur_ce   = float('inf')
best_dur_epoch = 0
bad_epochs = 0
t_global = time.monotonic()

print(f"\nRound 1: max 6 epochs, patience=2, budget=600s")
print(f"Output: {DUR_RUN_DIR}")

for epoch in range(1, 7):
    if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
        print("Budget reached."); break

    dur_model.train()
    loss_sum = n_seen = 0
    for cp_ids, err_idx, tgt, attn in dur_train_loader:
        if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
            break
        cp_ids, err_idx, tgt, attn = (
            x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
        dur_optimizer.zero_grad(set_to_none=True)
        logits = dur_model(cp_ids, err_idx, attn)
        loss = dur_loss_fn(logits, tgt)
        loss.backward()
        dur_optimizer.step()
        loss_sum += loss.item() * tgt.size(0)
        n_seen += tgt.size(0)

    if n_seen == 0:
        break
    dev = dur_eval(dur_model, dur_dev_loader)

    # top-1 primary; CE tiebreaker
    improved = (dev['top1_acc'] > best_dur_top1 or
                (dev['top1_acc'] == best_dur_top1 and dev['ce'] < best_dur_ce))

    row = {'epoch': epoch, 'train_loss': loss_sum / n_seen,
           'dev_top1': dev['top1_acc'], 'dev_top3': dev['top3_acc'],
           'dev_ce': dev['ce'],
           'correct_top1': dev['top1'], 'correct_top3': dev['top3'],
           'dev_n': dev['n'],
           'elapsed_s': time.monotonic() - t_global}
    dur_metrics_r1.append(row)

    if improved:
        best_dur_top1 = dev['top1_acc']
        best_dur_ce   = dev['ce']
        best_dur_epoch = epoch
        bad_epochs = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': {k: v.cpu()
                                 for k, v in dur_model.state_dict().items()},
            'optimizer_state_dict': dur_optimizer.state_dict(),
            'best_dev_top1': best_dur_top1,
            'metrics': dur_metrics_r1,
            'n_classes': N_DUR_CLASSES,
            'dur_label_map': dur_label_map,
        }, DUR_RUN_DIR / 'best_checkpoint.pt')
    else:
        bad_epochs += 1

    with open(DUR_RUN_DIR / 'metrics.json', 'w') as f:
        json.dump(dur_metrics_r1, f, indent=2)

    mark = ' *best*' if improved else ''
    print(f"  Epoch {epoch}: loss={loss_sum/n_seen:.4f}  "
          f"dev={dev['top1']}/{dev['n']} ({dev['top1_acc']:.2%})  "
          f"top-3={dev['top3_acc']:.2%}  CE={dev['ce']:.3f}{mark}")

    if bad_epochs >= 2:
        print(f"  Early stop: 2 epochs without improvement.")
        break

# ── Restore best R1 ─────────────────────────────────────────
_ckpt = torch.load(DUR_RUN_DIR / 'best_checkpoint.pt',
                    map_location='cpu', weights_only=False)
dur_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
dur_model.to(DEVICE); dur_model.eval()
del _ckpt

r1_dev = dur_eval(dur_model, dur_dev_loader)
with open(DUR_RUN_DIR / 'final_dev_metrics.json', 'w') as f:
    json.dump({'best_epoch': best_dur_epoch, **r1_dev,
               'mfc_hits': dur_mfc_hits}, f, indent=2)

r1_elapsed = time.monotonic() - t_global
print(f"\nRound 1 best: epoch {best_dur_epoch}")
print(f"  Top-1: {r1_dev['top1']}/{r1_dev['n']} = {r1_dev['top1_acc']:.2%}")
print(f"  Top-3: {r1_dev['top3']}/{r1_dev['n']} = {r1_dev['top3_acc']:.2%}")
print(f"  CE:    {r1_dev['ce']:.3f}")
print(f"  MFC:   {dur_mfc_hits}/{r1_dev['n']} = {dur_mfc_hits/r1_dev['n']:.2%}")
print(f"  Time:  {r1_elapsed:.0f}s")

In [ ]:
# ── Duration Round 2 (conditional) ───────────────────────────────
# Trigger if R1 didn't beat MFC, or dev top-1 dropped ≥5% from peak.
# Modification: differential LR — backbone 2e-5, head 1e-4.

r1_beat_mfc = r1_dev['top1'] > dur_mfc_hits
r1_overfit  = (len(dur_metrics_r1) >= 3 and
               dur_metrics_r1[-1]['dev_top1'] < max(r['dev_top1'] for r in dur_metrics_r1) * 0.95)
need_r2 = not r1_beat_mfc or r1_overfit
budget_left = MAX_SECONDS_TOTAL - r1_elapsed

# Initialise R2 variables (used by results cell regardless)
r2_dev = None
DUR_RUN_DIR_R2 = None
dur_metrics_r2 = []
best_r2_epoch = 0

if need_r2 and budget_left > 30:
    reasons = []
    if not r1_beat_mfc:
        reasons.append(f"R1 ({r1_dev['top1']}) <= MFC ({dur_mfc_hits})")
    if r1_overfit:
        reasons.append("dev top-1 dropped >=5% from peak")
    print(f"Round 2 triggered: {'; '.join(reasons)}")
    print(f"Modification: backbone LR 2e-5, head LR 1e-4 (was uniform 1e-4)")

    # Reload R1 best
    _ckpt = torch.load(DUR_RUN_DIR / 'best_checkpoint.pt',
                        map_location='cpu', weights_only=False)
    dur_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
    dur_model.to(DEVICE)
    del _ckpt

    backbone_lr, head_lr = 2e-5, 1e-4
    dur_optimizer_r2 = torch.optim.Adam([
        {'params': [p for li in (10, 11)
                    for p in dur_model.backbone.note_bert1.layers[li].parameters()
                    if p.requires_grad],
         'lr': backbone_lr},
        {'params': list(dur_model.classifier.parameters()), 'lr': head_lr},
    ])

    DUR_RUN_DIR_R2 = ML_DIR / 'data' / 'runs' / (
        'm2bert_dur_r2_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    DUR_RUN_DIR_R2.mkdir(parents=True, exist_ok=True)
    with open(DUR_RUN_DIR_R2 / 'config.json', 'w') as f:
        json.dump({'round': 2, 'backbone_lr': backbone_lr, 'head_lr': head_lr,
                   'reason': reasons, 'source': str(DUR_RUN_DIR),
                   'n_classes': N_DUR_CLASSES}, f, indent=2)

    best_r2_top1 = best_dur_top1
    best_r2_ce   = best_dur_ce
    bad_r2 = 0
    t_r2 = time.monotonic()
    print(f"Output: {DUR_RUN_DIR_R2}")

    for epoch in range(1, 7):
        if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
            print("  Budget reached."); break

        dur_model.train()
        loss_sum = n_seen = 0
        for cp_ids, err_idx, tgt, attn in dur_train_loader:
            if time.monotonic() - t_global >= MAX_SECONDS_TOTAL:
                break
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
            dur_optimizer_r2.zero_grad(set_to_none=True)
            logits = dur_model(cp_ids, err_idx, attn)
            loss = dur_loss_fn(logits, tgt)
            loss.backward()
            dur_optimizer_r2.step()
            loss_sum += loss.item() * tgt.size(0)
            n_seen += tgt.size(0)

        if n_seen == 0:
            break
        dev = dur_eval(dur_model, dur_dev_loader)
        improved = (dev['top1_acc'] > best_r2_top1 or
                    (dev['top1_acc'] == best_r2_top1 and dev['ce'] < best_r2_ce))

        row = {'epoch': epoch, 'train_loss': loss_sum / n_seen,
               'dev_top1': dev['top1_acc'], 'dev_top3': dev['top3_acc'],
               'dev_ce': dev['ce'],
               'correct_top1': dev['top1'], 'correct_top3': dev['top3'],
               'dev_n': dev['n'],
               'elapsed_s': time.monotonic() - t_r2}
        dur_metrics_r2.append(row)

        if improved:
            best_r2_top1 = dev['top1_acc']
            best_r2_ce   = dev['ce']
            best_r2_epoch = epoch
            bad_r2 = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': {k: v.cpu()
                                     for k, v in dur_model.state_dict().items()},
                'optimizer_state_dict': dur_optimizer_r2.state_dict(),
                'best_dev_top1': best_r2_top1,
                'metrics': dur_metrics_r2,
                'n_classes': N_DUR_CLASSES,
                'dur_label_map': dur_label_map,
                'round': 2, 'backbone_lr': backbone_lr, 'head_lr': head_lr,
            }, DUR_RUN_DIR_R2 / 'best_checkpoint.pt')
        else:
            bad_r2 += 1

        with open(DUR_RUN_DIR_R2 / 'metrics.json', 'w') as f:
            json.dump(dur_metrics_r2, f, indent=2)

        mark = ' *best*' if improved else ''
        print(f"  R2 Epoch {epoch}: loss={loss_sum/n_seen:.4f}  "
              f"dev={dev['top1']}/{dev['n']} ({dev['top1_acc']:.2%})  "
              f"top-3={dev['top3_acc']:.2%}  CE={dev['ce']:.3f}{mark}")

        if bad_r2 >= 2:
            print("  Early stop."); break

    # Restore R2 best
    if (DUR_RUN_DIR_R2 / 'best_checkpoint.pt').exists():
        _ckpt = torch.load(DUR_RUN_DIR_R2 / 'best_checkpoint.pt',
                            map_location='cpu', weights_only=False)
        dur_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
        dur_model.to(DEVICE); dur_model.eval()
        del _ckpt
        r2_dev = dur_eval(dur_model, dur_dev_loader)
        with open(DUR_RUN_DIR_R2 / 'final_dev_metrics.json', 'w') as f:
            json.dump({'best_epoch': best_r2_epoch, **r2_dev}, f, indent=2)
        print(f"\n  R2 best: epoch {best_r2_epoch}  "
              f"top-1={r2_dev['top1']}/{r2_dev['n']} ({r2_dev['top1_acc']:.2%})")
    else:
        print("  R2 produced no checkpoint (budget exhausted mid-epoch).")

elif need_r2:
    print(f"Round 2 needed but budget exhausted ({budget_left:.0f}s left).")
else:
    print(f"Round 2 not needed: R1 beats MFC and no overfitting.")

In [ ]:
# ── Duration Results & Demo ──────────────────────────────────────

# Pick the best model (R2 if it exists and is better, else R1)
if r2_dev is not None and r2_dev['top1'] > r1_dev['top1']:
    best_dur_run = DUR_RUN_DIR_R2
    best_dur_final = r2_dev
    best_dur_src = 'R2'
else:
    best_dur_run = DUR_RUN_DIR
    best_dur_final = r1_dev
    best_dur_src = 'R1'

# ── Comparison table ─────────────────────────────────────────
n_dev = best_dur_final['n']
print(f"{'='*72}")
print(f"  DURATION CORRECTION — COMPARISON TABLE  (dev N={n_dev})")
print(f"{'='*72}")
print(f"  {'Method':<28} {'Top-1':>10} {'Top-3':>10} {'Dev CE':>10}")
print(f"  {'─'*68}")
mfc_pct = f"{dur_mfc_hits/n_dev:.2%}"
print(f"  {'MFC ('+dur_mfc_label+' QL)':<28} "
      f"{dur_mfc_hits:>4}/{n_dev:<4}     {'—':>10} {'—':>10}")
r1_t1 = f"{r1_dev['top1']}/{n_dev}"
r1_t3 = f"{r1_dev['top3']}/{n_dev}"
print(f"  {'R1 (epoch '+str(best_dur_epoch)+', LR=1e-4)':<28} "
      f"{r1_t1:>10} {r1_t3:>10} {r1_dev['ce']:>10.3f}")
if r2_dev is not None:
    r2_t1 = f"{r2_dev['top1']}/{n_dev}"
    r2_t3 = f"{r2_dev['top3']}/{n_dev}"
    print(f"  {'R2 (epoch '+str(best_r2_epoch)+', diff LR)':<28} "
          f"{r2_t1:>10} {r2_t3:>10} {r2_dev['ce']:>10.3f}")
print(f"  {'─'*68}")

# ── Training curve ───────────────────────────────────────────
print(f"\nR1 training curve:")
for r in dur_metrics_r1:
    print(f"  Epoch {r['epoch']}: train_loss={r['train_loss']:.4f}  "
          f"dev_top1={r['correct_top1']}/{r['dev_n']} ({r['dev_top1']:.2%})  "
          f"dev_top3={r['dev_top3']:.2%}  CE={r['dev_ce']:.3f}  "
          f"time={r['elapsed_s']:.0f}s")
if dur_metrics_r2:
    print(f"\nR2 training curve:")
    for r in dur_metrics_r2:
        print(f"  Epoch {r['epoch']}: train_loss={r['train_loss']:.4f}  "
              f"dev_top1={r['correct_top1']}/{r['dev_n']} ({r['dev_top1']:.2%})  "
              f"dev_top3={r['dev_top3']:.2%}  CE={r['dev_ce']:.3f}  "
              f"time={r['elapsed_s']:.0f}s")

# ── Save dev predictions ─────────────────────────────────────
dur_model.eval()
dev_preds = []
with torch.inference_mode():
    for cp_ids, err_idx, tgt, attn in dur_dev_loader:
        cp_ids, err_idx, tgt, attn = (
            x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
        logits = dur_model(cp_ids, err_idx, attn)
        preds = logits.argmax(1).cpu()
        for j in range(tgt.size(0)):
            dev_preds.append({
                'target': dur_id2label[tgt[j].item()],
                'predicted': dur_id2label[preds[j].item()],
                'correct': bool(preds[j] == tgt[j]),
            })
with open(best_dur_run / 'dev_predictions.json', 'w') as f:
    json.dump(dev_preds, f, indent=2)

# ── 6-sample demo ────────────────────────────────────────────
rng_d = random.Random(42)
d_idx = sorted(rng_d.sample(range(len(dur_dev_cp)), 6))
d_ex  = [dur_dev_cp[i] for i in d_idx]
d_det = [dur_dev_details[i] for i in d_idx]

d_loader = DataLoader(CPDataset(d_ex), batch_size=6,
                      shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(d_loader))
with torch.inference_mode():
    logits = dur_model(cp_ids.to(DEVICE), err_idx.to(DEVICE), attn.to(DEVICE))
    probs = logits.softmax(-1).cpu()
    topk = probs.topk(min(5, N_DUR_CLASSES), dim=-1)

print(f"\n{'='*90}")
print(f"  6-SAMPLE DEMO (seed=42)")
print(f"{'='*90}")
print(f"  {'Idx':>4} {'Sample':>28}  {'Error':>8} {'Truth':>8} {'Pred':>8} {'OK':>3}  Candidates")
print(f"  {'─'*86}")
for b in range(6):
    tgt_id  = tgt[b].item()
    pred_id = topk.indices[b][0].item()
    ok = ' ok' if pred_id == tgt_id else '  X'
    cands = [dur_id2label[topk.indices[b][k].item()]
             for k in range(min(5, N_DUR_CLASSES))]
    d = d_det[b]
    print(f"  {d_idx[b]:4d} {d['sample_id']:>28}  "
          f"{d['repl_ql']:>8} {d['true_ql']:>8} "
          f"{dur_id2label[pred_id]:>8}{ok}  {cands}")

# ── Summary ──────────────────────────────────────────────────
beat_mfc = best_dur_final['top1'] > dur_mfc_hits
r2_ran   = r2_dev is not None
total_s  = time.monotonic() - t_global

print(f"""
{'='*72}
DURATION CORRECTION — SUMMARY
{'='*72}
Task:    Predict masked note duration (exact quarterLength, {N_DUR_CLASSES} classes).
Data:    {len(dur_train_cp)} train / {len(dur_dev_cp)} dev examples.
         Grace notes excluded; rests excluded; OOV: {dur_dev_stats['oov_durations']}.
Model:   M2BERT backbone (frozen 0-9) + Linear({N_DUR_CLASSES}).
MFC:     {dur_mfc_hits}/{n_dev} ({dur_mfc_hits/n_dev:.2%}) — always predict '{dur_mfc_label}' QL.
R1:      {r1_dev['top1']}/{n_dev} ({r1_dev['top1_acc']:.2%}), epoch {best_dur_epoch}, LR=1e-4.
R2:      {'Not run' if not r2_ran else f"{r2_dev['top1']}/{n_dev} ({r2_dev['top1_acc']:.2%}), epoch {best_r2_epoch}, diff LR."} {'('+best_dur_src+' selected)' if r2_ran else ''}
Best:    {best_dur_final['top1']}/{n_dev} ({best_dur_final['top1_acc']:.2%}) [{best_dur_src}]
         {'Beats MFC.' if beat_mfc else 'Does NOT beat MFC.'}
Time:    {total_s:.0f}s total training.

Next concrete improvement:
  The duration label space is small ({N_DUR_CLASSES} classes) and highly imbalanced
  (MFC covers {dur_mfc_hits/n_dev:.0%} of dev). The CP Position attribute quantises
  onset to 16th-note grid (0-15), losing sub-beat precision that distinguishes
  tuplet from straight durations. A concrete next step: add an explicit
  onset-in-ticks attribute (finer than 16th grid) to the CP input, giving the
  model access to the rhythmic context that the current 4-bit position discards.
{'='*72}
""")

## Joint Pitch + Duration Correction

Shared M2BERT backbone, two prediction heads (52-class pitch, 28-class duration).  
Train with 1:1 interleaved batches; checkpoint by average dev top-1.

**Single-task baselines** (to preserve, not retrain):
- Pitch: 312/499 (62.53%), top-5 427/499 (85.57%)
- Duration: 434/499 (86.97%), top-3 481/499 (96.39%)

In [ ]:
# ── Joint Model: Setup + Round 1 Training ────────────────────────
# Shared backbone, two heads. 1:1 interleaved pitch/duration batches.
# Checkpoint by mean(pitch_dev_top1, dur_dev_top1).

from itertools import zip_longest

# ── Model ────────────────────────────────────────────────────
class M2BERTJointCorrection(nn.Module):
    """Shared backbone with separate pitch and duration heads."""
    def __init__(self, backbone, n_pitch, n_dur):
        super().__init__()
        self.backbone   = backbone
        self.pitch_head = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_pitch))
        self.dur_head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, n_dur))

    def forward(self, cp_ids, error_note_idx, attention_mask=None, task='pitch'):
        hidden = self.backbone(cp_ids, attention_mask)
        err_h  = hidden[torch.arange(hidden.size(0)), error_note_idx]
        if task == 'pitch':
            return self.pitch_head(err_h)
        return self.dur_head(err_h)

# Fresh pretrained backbone
_jbb = MidiModernBERT()
_ckpt = torch.load(PRETRAINED / 'model_best.ckpt', map_location='cpu',
                    weights_only=False)
_jbb.load_state_dict(_ckpt['state_dict'], strict=True)
print(f'Backbone loaded (pretrained epoch {_ckpt["epoch"]})')
del _ckpt

joint_model = M2BERTJointCorrection(_jbb, N_CLASSES, N_DUR_CLASSES)
joint_model.to(DEVICE)

# Freeze layers 0-9, train layers 10-11 + both heads
for p in joint_model.parameters():
    p.requires_grad_(False)
for li in (10, 11):
    for p in joint_model.backbone.note_bert1.layers[li].parameters():
        p.requires_grad_(True)
for p in joint_model.pitch_head.parameters():
    p.requires_grad_(True)
for p in joint_model.dur_head.parameters():
    p.requires_grad_(True)

n_tp = sum(p.numel() for p in joint_model.parameters() if p.requires_grad)
n_fp = sum(p.numel() for p in joint_model.parameters() if not p.requires_grad)
print(f'Trainable: {n_tp:,}  Frozen: {n_fp:,}')
print(f'Heads: pitch={N_CLASSES}, duration={N_DUR_CLASSES}')

JOINT_LR = 1e-4
joint_optimizer = torch.optim.Adam(
    [p for p in joint_model.parameters() if p.requires_grad], lr=JOINT_LR)
joint_loss_fn = nn.CrossEntropyLoss()

JOINT_RUN = ML_DIR / 'data' / 'runs' / (
    'm2bert_joint_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
JOINT_RUN.mkdir(parents=True, exist_ok=True)

with open(JOINT_RUN / 'config.json', 'w') as f:
    json.dump({'lr': JOINT_LR, 'batch_size': 32, 'max_epochs': 6,
               'patience': 2, 'n_pitch_classes': N_CLASSES,
               'n_dur_classes': N_DUR_CLASSES,
               'trainable_layers': '10-11 + both heads',
               'pitch_train_n': len(train_cp), 'dur_train_n': len(dur_train_cp),
               'pitch_dev_n': len(dev_cp), 'dur_dev_n': len(dur_dev_cp),
               'interleave': '1:1'}, f, indent=2)
with open(JOINT_RUN / 'pitch_label_map.json', 'w') as f:
    json.dump(pitch_classes, f, indent=2)
with open(JOINT_RUN / 'dur_label_map.json', 'w') as f:
    json.dump(dur_label_map, f, indent=2)

# ── Data loaders ─────────────────────────────────────────────
jt_pitch_loader = DataLoader(CPDataset(train_cp), batch_size=32,
                             shuffle=True, collate_fn=collate_cp, num_workers=0)
jt_dur_loader   = DataLoader(CPDataset(dur_train_cp), batch_size=32,
                             shuffle=True, collate_fn=collate_cp, num_workers=0)
jt_pitch_dev    = DataLoader(CPDataset(dev_cp), batch_size=64,
                             shuffle=False, collate_fn=collate_cp, num_workers=0)
jt_dur_dev      = DataLoader(CPDataset(dur_dev_cp), batch_size=64,
                             shuffle=False, collate_fn=collate_cp, num_workers=0)

def joint_eval(mdl, pitch_loader, dur_loader):
    """Evaluate both heads separately. Returns (pitch_metrics, dur_metrics)."""
    mdl.eval()
    results = {}
    for task, loader, topk in [('pitch', pitch_loader, 5),
                                ('dur', dur_loader, 3)]:
        n = t1 = tk = 0; ce = 0.0
        with torch.inference_mode():
            for cp_ids, err_idx, tgt, attn in loader:
                cp_ids, err_idx, tgt, attn = (
                    x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
                logits = mdl(cp_ids, err_idx, attn, task=task)
                t1 += (logits.argmax(1) == tgt).sum().item()
                tk += (logits.topk(min(topk, logits.size(1)), dim=1
                       ).indices == tgt[:, None]).any(1).sum().item()
                ce += F.cross_entropy(logits, tgt, reduction='sum').item()
                n += tgt.size(0)
        results[task] = {'n': n, 'top1': t1, f'top{topk}': tk,
                         'top1_acc': t1 / n, f'top{topk}_acc': tk / n,
                         'ce': ce / n}
    return results['pitch'], results['dur']

# ── Training Round 1 ────────────────────────────────────────
JT_BUDGET = 600
jt_metrics = []
best_jt_score = 0.0       # mean(pitch_top1_acc, dur_top1_acc)
best_jt_ce    = float('inf')
best_jt_epoch = 0
jt_bad = 0
jt_t0 = time.monotonic()

print(f"\nJoint R1: max 6 epochs, patience=2, budget={JT_BUDGET}s")
print(f"Output: {JOINT_RUN}")

for epoch in range(1, 7):
    if time.monotonic() - jt_t0 >= JT_BUDGET:
        print("Budget reached."); break

    joint_model.train()
    p_loss_sum = p_n = d_loss_sum = d_n = 0

    # Interleave: alternate pitch batch, duration batch
    for p_batch, d_batch in zip_longest(jt_pitch_loader, jt_dur_loader):
        if time.monotonic() - jt_t0 >= JT_BUDGET:
            break

        # Pitch step
        if p_batch is not None:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in p_batch)
            joint_optimizer.zero_grad(set_to_none=True)
            logits = joint_model(cp_ids, err_idx, attn, task='pitch')
            loss = joint_loss_fn(logits, tgt)
            loss.backward()
            joint_optimizer.step()
            p_loss_sum += loss.item() * tgt.size(0)
            p_n += tgt.size(0)

        if time.monotonic() - jt_t0 >= JT_BUDGET:
            break

        # Duration step
        if d_batch is not None:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in d_batch)
            joint_optimizer.zero_grad(set_to_none=True)
            logits = joint_model(cp_ids, err_idx, attn, task='dur')
            loss = joint_loss_fn(logits, tgt)
            loss.backward()
            joint_optimizer.step()
            d_loss_sum += loss.item() * tgt.size(0)
            d_n += tgt.size(0)

    if p_n == 0 and d_n == 0:
        break

    p_dev, d_dev = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
    score = (p_dev['top1_acc'] + d_dev['top1_acc']) / 2
    avg_ce = (p_dev['ce'] + d_dev['ce']) / 2
    improved = (score > best_jt_score or
                (score == best_jt_score and avg_ce < best_jt_ce))

    row = {
        'epoch': epoch,
        'pitch_train_loss': p_loss_sum / max(p_n, 1),
        'dur_train_loss': d_loss_sum / max(d_n, 1),
        'pitch_dev_top1': p_dev['top1_acc'],
        'pitch_dev_top5': p_dev['top5_acc'],
        'pitch_dev_ce': p_dev['ce'],
        'pitch_correct': p_dev['top1'],
        'dur_dev_top1': d_dev['top1_acc'],
        'dur_dev_top3': d_dev['top3_acc'],
        'dur_dev_ce': d_dev['ce'],
        'dur_correct': d_dev['top1'],
        'avg_top1': score,
        'avg_ce': avg_ce,
        'pitch_n': p_dev['n'], 'dur_n': d_dev['n'],
        'elapsed_s': time.monotonic() - jt_t0,
    }
    jt_metrics.append(row)

    if improved:
        best_jt_score = score
        best_jt_ce    = avg_ce
        best_jt_epoch = epoch
        jt_bad = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': {k: v.cpu()
                                 for k, v in joint_model.state_dict().items()},
            'optimizer_state_dict': joint_optimizer.state_dict(),
            'best_avg_top1': best_jt_score,
            'metrics': jt_metrics,
            'n_pitch_classes': N_CLASSES,
            'n_dur_classes': N_DUR_CLASSES,
            'pitch_classes': pitch_classes,
            'dur_label_map': dur_label_map,
        }, JOINT_RUN / 'best_checkpoint.pt')
    else:
        jt_bad += 1

    with open(JOINT_RUN / 'metrics.json', 'w') as f:
        json.dump(jt_metrics, f, indent=2)

    mark = ' *best*' if improved else ''
    print(f"  Ep {epoch}: P={p_dev['top1']}/{p_dev['n']}({p_dev['top1_acc']:.2%}) "
          f"D={d_dev['top1']}/{d_dev['n']}({d_dev['top1_acc']:.2%}) "
          f"avg={score:.4f} "
          f"pCE={p_dev['ce']:.3f} dCE={d_dev['ce']:.3f} "
          f"t={row['elapsed_s']:.0f}s{mark}")

    if jt_bad >= 2:
        print("  Early stop."); break

# ── Restore best R1 ─────────────────────────────────────────
_ckpt = torch.load(JOINT_RUN / 'best_checkpoint.pt',
                    map_location='cpu', weights_only=False)
joint_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
joint_model.to(DEVICE); joint_model.eval()
del _ckpt

jt_r1_p, jt_r1_d = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
with open(JOINT_RUN / 'final_dev_metrics.json', 'w') as f:
    json.dump({'best_epoch': best_jt_epoch,
               'pitch': jt_r1_p, 'dur': jt_r1_d}, f, indent=2)

jt_r1_elapsed = time.monotonic() - jt_t0
print(f"\nR1 best epoch {best_jt_epoch}:")
print(f"  Pitch: {jt_r1_p['top1']}/{jt_r1_p['n']} = {jt_r1_p['top1_acc']:.2%}  "
      f"top-5={jt_r1_p['top5_acc']:.2%}  CE={jt_r1_p['ce']:.3f}")
print(f"  Dur:   {jt_r1_d['top1']}/{jt_r1_d['n']} = {jt_r1_d['top1_acc']:.2%}  "
      f"top-3={jt_r1_d['top3_acc']:.2%}  CE={jt_r1_d['ce']:.3f}")
print(f"  Avg top-1: {(jt_r1_p['top1_acc']+jt_r1_d['top1_acc'])/2:.4f}")
print(f"  Time: {jt_r1_elapsed:.0f}s")

In [ ]:
# ── Joint Round 2 (conditional) ──────────────────────────────────
# Trigger if pitch OR duration dev top-1 dropped >3pp vs single-task baseline.
# One modification only, based on which task regressed more.

PITCH_BASELINE = 0.6253   # 312/499
DUR_BASELINE   = 0.8697   # 434/499

p_delta = jt_r1_p['top1_acc'] - PITCH_BASELINE
d_delta = jt_r1_d['top1_acc'] - DUR_BASELINE
p_regressed = p_delta < -0.03
d_regressed = d_delta < -0.03
need_jt_r2  = p_regressed or d_regressed
jt_budget_left = JT_BUDGET - jt_r1_elapsed

# Initialise R2 variables
jt_r2_p = jt_r2_d = None
JOINT_RUN_R2 = None
jt_metrics_r2 = []
best_jt_r2_epoch = 0

if need_jt_r2 and jt_budget_left > 30:
    # Decide modification
    if p_regressed and not d_regressed:
        mod_desc = "Pitch regressed more → upweight pitch loss 2:1"
        p_weight, d_weight = 2.0, 1.0
        bb_lr, head_lr = JOINT_LR, JOINT_LR
    elif d_regressed and not p_regressed:
        mod_desc = "Duration regressed more → upweight duration loss 2:1"
        p_weight, d_weight = 1.0, 2.0
        bb_lr, head_lr = JOINT_LR, JOINT_LR
    else:
        mod_desc = "Both regressed → lower backbone LR to 2e-5, heads stay 1e-4"
        p_weight, d_weight = 1.0, 1.0
        bb_lr, head_lr = 2e-5, 1e-4

    print(f"R2 triggered: pitch Δ={p_delta:+.2%}, dur Δ={d_delta:+.2%}")
    print(f"  Modification: {mod_desc}")

    # Reload R1 best
    _ckpt = torch.load(JOINT_RUN / 'best_checkpoint.pt',
                        map_location='cpu', weights_only=False)
    joint_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
    joint_model.to(DEVICE)
    del _ckpt

    if bb_lr != head_lr:
        jt_opt_r2 = torch.optim.Adam([
            {'params': [p for li in (10, 11)
                        for p in joint_model.backbone.note_bert1.layers[li].parameters()
                        if p.requires_grad], 'lr': bb_lr},
            {'params': list(joint_model.pitch_head.parameters()) +
                       list(joint_model.dur_head.parameters()), 'lr': head_lr},
        ])
    else:
        jt_opt_r2 = torch.optim.Adam(
            [p for p in joint_model.parameters() if p.requires_grad], lr=JOINT_LR)

    p_loss_fn_r2 = nn.CrossEntropyLoss()
    d_loss_fn_r2 = nn.CrossEntropyLoss()

    JOINT_RUN_R2 = ML_DIR / 'data' / 'runs' / (
        'm2bert_joint_r2_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    JOINT_RUN_R2.mkdir(parents=True, exist_ok=True)
    with open(JOINT_RUN_R2 / 'config.json', 'w') as f:
        json.dump({'round': 2, 'modification': mod_desc,
                   'p_weight': p_weight, 'd_weight': d_weight,
                   'backbone_lr': bb_lr, 'head_lr': head_lr,
                   'source': str(JOINT_RUN)}, f, indent=2)

    best_r2_score = best_jt_score
    best_r2_ce    = best_jt_ce
    bad_r2 = 0
    t_r2 = time.monotonic()
    print(f"  Output: {JOINT_RUN_R2}")

    for epoch in range(1, 7):
        if time.monotonic() - jt_t0 >= JT_BUDGET:
            print("  Budget reached."); break

        joint_model.train()
        p_ls = p_ns = d_ls = d_ns = 0

        for p_batch, d_batch in zip_longest(jt_pitch_loader, jt_dur_loader):
            if time.monotonic() - jt_t0 >= JT_BUDGET:
                break
            if p_batch is not None:
                cp_ids, err_idx, tgt, attn = (
                    x.to(DEVICE) for x in p_batch)
                jt_opt_r2.zero_grad(set_to_none=True)
                logits = joint_model(cp_ids, err_idx, attn, task='pitch')
                loss = p_loss_fn_r2(logits, tgt) * p_weight
                loss.backward(); jt_opt_r2.step()
                p_ls += loss.item() / p_weight * tgt.size(0)
                p_ns += tgt.size(0)
            if time.monotonic() - jt_t0 >= JT_BUDGET:
                break
            if d_batch is not None:
                cp_ids, err_idx, tgt, attn = (
                    x.to(DEVICE) for x in d_batch)
                jt_opt_r2.zero_grad(set_to_none=True)
                logits = joint_model(cp_ids, err_idx, attn, task='dur')
                loss = d_loss_fn_r2(logits, tgt) * d_weight
                loss.backward(); jt_opt_r2.step()
                d_ls += loss.item() / d_weight * tgt.size(0)
                d_ns += tgt.size(0)

        if p_ns == 0 and d_ns == 0:
            break

        p_dev, d_dev = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
        score = (p_dev['top1_acc'] + d_dev['top1_acc']) / 2
        avg_ce = (p_dev['ce'] + d_dev['ce']) / 2
        improved = (score > best_r2_score or
                    (score == best_r2_score and avg_ce < best_r2_ce))

        row = {
            'epoch': epoch,
            'pitch_train_loss': p_ls / max(p_ns, 1),
            'dur_train_loss': d_ls / max(d_ns, 1),
            'pitch_dev_top1': p_dev['top1_acc'],
            'pitch_dev_top5': p_dev.get('top5_acc', 0),
            'pitch_dev_ce': p_dev['ce'],
            'pitch_correct': p_dev['top1'],
            'dur_dev_top1': d_dev['top1_acc'],
            'dur_dev_top3': d_dev.get('top3_acc', 0),
            'dur_dev_ce': d_dev['ce'],
            'dur_correct': d_dev['top1'],
            'avg_top1': score, 'avg_ce': avg_ce,
            'pitch_n': p_dev['n'], 'dur_n': d_dev['n'],
            'elapsed_s': time.monotonic() - t_r2,
        }
        jt_metrics_r2.append(row)

        if improved:
            best_r2_score = score; best_r2_ce = avg_ce
            best_jt_r2_epoch = epoch; bad_r2 = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': {k: v.cpu()
                                     for k, v in joint_model.state_dict().items()},
                'optimizer_state_dict': jt_opt_r2.state_dict(),
                'best_avg_top1': best_r2_score,
                'metrics': jt_metrics_r2,
                'n_pitch_classes': N_CLASSES,
                'n_dur_classes': N_DUR_CLASSES,
                'pitch_classes': pitch_classes,
                'dur_label_map': dur_label_map,
                'round': 2, 'modification': mod_desc,
            }, JOINT_RUN_R2 / 'best_checkpoint.pt')
        else:
            bad_r2 += 1

        with open(JOINT_RUN_R2 / 'metrics.json', 'w') as f:
            json.dump(jt_metrics_r2, f, indent=2)

        mark = ' *best*' if improved else ''
        print(f"  R2 Ep {epoch}: P={p_dev['top1']}/{p_dev['n']}({p_dev['top1_acc']:.2%}) "
              f"D={d_dev['top1']}/{d_dev['n']}({d_dev['top1_acc']:.2%}) "
              f"avg={score:.4f}{mark}")

        if bad_r2 >= 2:
            print("  Early stop."); break

    # Restore R2 best
    if JOINT_RUN_R2 and (JOINT_RUN_R2 / 'best_checkpoint.pt').exists():
        _ckpt = torch.load(JOINT_RUN_R2 / 'best_checkpoint.pt',
                            map_location='cpu', weights_only=False)
        joint_model.load_state_dict(_ckpt['model_state_dict'], strict=True)
        joint_model.to(DEVICE); joint_model.eval()
        del _ckpt
        jt_r2_p, jt_r2_d = joint_eval(joint_model, jt_pitch_dev, jt_dur_dev)
        with open(JOINT_RUN_R2 / 'final_dev_metrics.json', 'w') as f:
            json.dump({'best_epoch': best_jt_r2_epoch,
                       'pitch': jt_r2_p, 'dur': jt_r2_d}, f, indent=2)
        print(f"\n  R2 best ep {best_jt_r2_epoch}: "
              f"P={jt_r2_p['top1']}/{jt_r2_p['n']}({jt_r2_p['top1_acc']:.2%})  "
              f"D={jt_r2_d['top1']}/{jt_r2_d['n']}({jt_r2_d['top1_acc']:.2%})")

elif need_jt_r2:
    print(f"R2 needed but budget exhausted ({jt_budget_left:.0f}s left).")
else:
    print(f"R2 not needed: pitch Δ={p_delta:+.2%}, dur Δ={d_delta:+.2%} "
          f"(threshold: -3pp)")

In [ ]:
# ── Joint Results, Demo & Summary ────────────────────────────────

# Select best run (R2 if better avg, else R1)
if jt_r2_p is not None:
    r2_avg = (jt_r2_p['top1_acc'] + jt_r2_d['top1_acc']) / 2
    r1_avg = (jt_r1_p['top1_acc'] + jt_r1_d['top1_acc']) / 2
    if r2_avg > r1_avg:
        jt_best_p, jt_best_d = jt_r2_p, jt_r2_d
        jt_best_src = 'R2'
        jt_best_run = JOINT_RUN_R2
    else:
        jt_best_p, jt_best_d = jt_r1_p, jt_r1_d
        jt_best_src = 'R1'
        jt_best_run = JOINT_RUN
else:
    jt_best_p, jt_best_d = jt_r1_p, jt_r1_d
    jt_best_src = 'R1'
    jt_best_run = JOINT_RUN

# ── Comparison table ─────────────────────────────────────────
print(f"{'='*80}")
print(f"  JOINT vs SINGLE-TASK COMPARISON")
print(f"{'='*80}")
print(f"  {'':30s} {'Pitch':>20s}   {'Duration':>20s}")
print(f"  {'':30s} {'top-1':>8s} {'top-5':>8s}   {'top-1':>8s} {'top-3':>8s}")
print(f"  {'─'*76}")

# Single-task baselines
print(f"  {'Single-task pitch':<30s} "
      f"{'312/499':>8s} {'427/499':>8s}   {'—':>8s} {'—':>8s}")
print(f"  {'Single-task duration':<30s} "
      f"{'—':>8s} {'—':>8s}   {'434/499':>8s} {'481/499':>8s}")

# Joint R1
r1p, r1d = jt_r1_p, jt_r1_d
p_t5_key = 'top5' if 'top5' in r1p else 'top5_acc'
d_t3_key = 'top3' if 'top3' in r1d else 'top3_acc'
r1p_t5 = r1p.get('top5', int(r1p.get('top5_acc', 0) * r1p['n']))
r1d_t3 = r1d.get('top3', int(r1d.get('top3_acc', 0) * r1d['n']))
print(f"  {'Joint R1 (ep '+str(best_jt_epoch)+')':<30s} "
      f"{r1p['top1']:>4d}/499  {r1p_t5:>4d}/499   "
      f"{r1d['top1']:>4d}/499  {r1d_t3:>4d}/499")

# Joint R2
if jt_r2_p is not None:
    r2p_t5 = jt_r2_p.get('top5', int(jt_r2_p.get('top5_acc', 0) * jt_r2_p['n']))
    r2d_t3 = jt_r2_d.get('top3', int(jt_r2_d.get('top3_acc', 0) * jt_r2_d['n']))
    print(f"  {'Joint R2 (ep '+str(best_jt_r2_epoch)+')':<30s} "
          f"{jt_r2_p['top1']:>4d}/499  {r2p_t5:>4d}/499   "
          f"{jt_r2_d['top1']:>4d}/499  {r2d_t3:>4d}/499")

print(f"  {'─'*76}")

# Delta vs baselines
bp = jt_best_p; bd = jt_best_d
p_d1 = bp['top1'] - 312; d_d1 = bd['top1'] - 434
print(f"  {'Selected: '+jt_best_src:<30s} "
      f"{bp['top1']:>4d}/499  "
      f"{bp.get('top5', int(bp.get('top5_acc',0)*bp['n'])):>4d}/499   "
      f"{bd['top1']:>4d}/499  "
      f"{bd.get('top3', int(bd.get('top3_acc',0)*bd['n'])):>4d}/499")
print(f"  {'Δ vs single-task':<30s} "
      f"{'%+d' % p_d1:>8s} {'':>8s}   {'%+d' % d_d1:>8s}")
print(f"{'='*80}")

# ── Training curves ──────────────────────────────────────────
print(f"\nR1 training curve:")
for r in jt_metrics:
    print(f"  Ep {r['epoch']}: pLoss={r['pitch_train_loss']:.4f} "
          f"dLoss={r['dur_train_loss']:.4f}  "
          f"P={r['pitch_correct']}/499({r['pitch_dev_top1']:.2%})  "
          f"D={r['dur_correct']}/499({r['dur_dev_top1']:.2%})  "
          f"avg={r['avg_top1']:.4f}  t={r['elapsed_s']:.0f}s")
if jt_metrics_r2:
    print(f"\nR2 training curve:")
    for r in jt_metrics_r2:
        print(f"  Ep {r['epoch']}: pLoss={r['pitch_train_loss']:.4f} "
              f"dLoss={r['dur_train_loss']:.4f}  "
              f"P={r['pitch_correct']}/499({r['pitch_dev_top1']:.2%})  "
              f"D={r['dur_correct']}/499({r['dur_dev_top1']:.2%})  "
              f"avg={r['avg_top1']:.4f}")

# ── Save dev predictions ─────────────────────────────────────
joint_model.eval()
for task, loader, id2label, fname in [
    ('pitch', jt_pitch_dev, id2pitch, 'pitch_dev_predictions.json'),
    ('dur',   jt_dur_dev,   dur_id2label, 'dur_dev_predictions.json'),
]:
    preds = []
    with torch.inference_mode():
        for cp_ids, err_idx, tgt, attn in loader:
            cp_ids, err_idx, tgt, attn = (
                x.to(DEVICE) for x in (cp_ids, err_idx, tgt, attn))
            logits = joint_model(cp_ids, err_idx, attn, task=task)
            pred = logits.argmax(1).cpu()
            for j in range(tgt.size(0)):
                preds.append({
                    'target': id2label[tgt[j].item()],
                    'predicted': id2label[pred[j].item()],
                    'correct': bool(pred[j] == tgt[j]),
                })
    with open(jt_best_run / fname, 'w') as f:
        json.dump(preds, f, indent=2)

# ── 6-sample demo (3 pitch + 3 duration) ────────────────────
print(f"\n{'='*90}")
print(f"  6-SAMPLE DEMO (3 pitch + 3 duration, seed=42)")
print(f"{'='*90}")

# Pitch demo
rng_jp = random.Random(42)
jp_idx = sorted(rng_jp.sample(range(len(dev_cp)), 3))
jp_ex  = [dev_cp[i] for i in jp_idx]
jp_raw = [dev_raw[i] for i in jp_idx]

jp_loader = DataLoader(CPDataset(jp_ex), batch_size=3,
                       shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(jp_loader))
with torch.inference_mode():
    logits = joint_model(cp_ids.to(DEVICE), err_idx.to(DEVICE),
                         attn.to(DEVICE), task='pitch')
    topk = logits.softmax(-1).cpu().topk(5, dim=-1)

print(f"\n  PITCH samples:")
print(f"  {'Idx':>4} {'Sample':>28}  {'Error':>5} {'Truth':>5} {'Pred':>5} {'OK':>3}  Top-5")
print(f"  {'─'*85}")
for b in range(3):
    pred_id = topk.indices[b][0].item()
    ok = ' ok' if pred_id == tgt[b].item() else '  X'
    cands = [id2pitch[topk.indices[b][k].item()] for k in range(5)]
    r = jp_raw[b]
    print(f"  {jp_idx[b]:4d} {r['sample_id']:>28}  "
          f"{r['replacement_token']:>5} {r['target_token']:>5} "
          f"{id2pitch[pred_id]:>5}{ok}  {cands}")

# Duration demo
rng_jd = random.Random(42)
jd_idx = sorted(rng_jd.sample(range(len(dur_dev_cp)), 3))
jd_ex  = [dur_dev_cp[i] for i in jd_idx]
jd_det = [dur_dev_details[i] for i in jd_idx]

jd_loader = DataLoader(CPDataset(jd_ex), batch_size=3,
                       shuffle=False, collate_fn=collate_cp)
cp_ids, err_idx, tgt, attn = next(iter(jd_loader))
with torch.inference_mode():
    logits = joint_model(cp_ids.to(DEVICE), err_idx.to(DEVICE),
                         attn.to(DEVICE), task='dur')
    topk = logits.softmax(-1).cpu().topk(min(5, N_DUR_CLASSES), dim=-1)

print(f"\n  DURATION samples:")
print(f"  {'Idx':>4} {'Sample':>28}  {'Error':>8} {'Truth':>8} {'Pred':>8} {'OK':>3}  Top candidates")
print(f"  {'─'*90}")
for b in range(3):
    pred_id = topk.indices[b][0].item()
    ok = ' ok' if pred_id == tgt[b].item() else '  X'
    cands = [dur_id2label[topk.indices[b][k].item()]
             for k in range(min(5, N_DUR_CLASSES))]
    d = jd_det[b]
    print(f"  {jd_idx[b]:4d} {d['sample_id']:>28}  "
          f"{d['repl_ql']:>8} {d['true_ql']:>8} "
          f"{dur_id2label[pred_id]:>8}{ok}  {cands}")

# ── Summary ──────────────────────────────────────────────────
total_jt_s = time.monotonic() - jt_t0
r2_ran = jt_r2_p is not None

print(f"""
{'='*80}
JOINT CORRECTION — SUMMARY
{'='*80}
Model:    Shared M2BERT backbone + pitch head (52) + duration head (28).
          Frozen layers 0-9; trainable layers 10-11 + both heads.
Training: 1:1 interleaved pitch/duration batches, each with mean CE loss.
          Checkpoint by mean(pitch_dev_top1, dur_dev_top1).

                    Pitch top-1      Duration top-1
  Single-task:      312/499 (62.53%)   434/499 (86.97%)
  Joint {jt_best_src}:       {bp['top1']}/499 ({bp['top1_acc']:.2%})   {bd['top1']}/499 ({bd['top1_acc']:.2%})
  Delta:            {'+' if p_d1>=0 else ''}{p_d1}               {'+' if d_d1>=0 else ''}{d_d1}

R2: {'Run — ' + (jt_metrics_r2[0].get('modification','') if jt_metrics_r2 else '') if r2_ran else 'Not triggered.'}
Time: {total_jt_s:.0f}s total.
Run:  {jt_best_run}

Next step: The joint model shares backbone representations across tasks.
  If one task's accuracy dropped, the next action is to try task-specific
  adapter layers (e.g., LoRA on layers 10-11 per task) so that the shared
  backbone provides a common foundation while each task retains its own
  fine-tuning capacity. This avoids the interference seen in uniform
  shared fine-tuning while preserving the parameter-efficiency benefit.
{'='*80}
""")